Note: Since GitHub does not execute JavaScript in notebooks, use link in the description to have interative plots

# 1. Data loading and setup

In [1]:
import pandas as pd
import numpy as np
import io
import requests
import lxml
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from scipy.spatial import distance
from IPython.display import HTML, Markdown, display
import warnings

pio.renderers.default = "notebook_connected"
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

### Concatenating test result for all months to one file

The `test_result` table is stored in separate monthly files. First, we filter the data by motorcycle classes (`['test_class_id']` = 1 or 2) and then concatenate the files into a single dataset. Despite the description of `['test_class_id']` in the MOT testing guide, some tests appear in class 1 but not in class 2.

In [2]:
# test_result_list = []
# for i in range(1, 13):
#     month = str(i).zfill(2)
#     file_path = fr"D:\IT\Datasets\MOT\MOT testing data results (2024)\test_result_2024{month}.csv"
#     try:
#         tr = pd.read_csv(
#             file_path, 
#             dtype={'test_id': 'object', 
#                    'vehicle_id': 'object', 
#                    'test_class_id': 'object', 
#                    'test_mileage': 'object', 
#                    'cylinder_capacity': 'object'}, 
#             on_bad_lines='skip'
#         )
#         tr = tr.loc[tr['test_class_id'].str.contains('1|2', regex=True)]
#         test_result_list.append(tr)
#     except FileNotFoundError:
#         print(f"File not found: {file_name}")
# if test_result_list:
#     test_result_full = pd.concat(test_result_list, ignore_index=True)
#     print("Successfully loaded and combined all files.")
# else:
#     print("No files were loaded.");

The concatenated file is saved for convenience.

In [3]:
# # continue with saved motorycle dataset
# test_result_full.to_csv(r'D:\IT\Datasets\MOT\test_result.csv')

### Concatenating 'Test Item' for all months

Next, we do the same for `test_item` table, filter motorcycles from `test_result` table.

In [4]:
# unique_test_id = test_result_full['test_id'].unique()
# test_item_list = []
# for i in range(1, 13):
#     month = str(i).zfill(2)
#     file_path = fr"D:\IT\Datasets\MOT\MOT Testing data failure item (2024)\test_item_2024{month}.csv"
#     try:
#         ti = pd.read_csv(file_path, 
#                          dtype={'test_id': 'object', 
#                                 'rfr_id': 'object', 
#                                 'mot_test_rfr_location_type_id': 'object'}
#                         )
#         ti = ti.loc[ti['test_id'].isin(unique_test_id)]
#         test_item_list.append(ti)
#     except FileNotFoundError:
#         print(f"File not found: {file_path}")
# if test_item_list:
#     test_item_full = pd.concat(test_item_list, ignore_index=True)
#     print("Successfully downloaded and combined all files.")
# else:
#     print("No files were loaded.");

Concatenated `test_item` table is saved to single csv file.

In [5]:
# test_item_full.to_csv(r"D:\IT\Datasets\MOT\test_item.csv")

Concatenated table along with other tables are loaded

In [6]:
test_result_original = pd.read_csv(r"D:\IT\Datasets\MOT\test_result.csv")
test_items_original = pd.read_csv(r"D:\IT\Datasets\MOT\test_item.csv")
item_detail_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_detail.csv", delimiter='|')
item_group_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_group.csv", delimiter='|')

In [7]:
# item_detail_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_detail.csv", header=None, names=range(10))
# first_col = item_detail_original[0].str.split('|', expand=True)
# header = first_col.iloc[0]
# item_detail_original = first_col.iloc[1:].reset_index(drop=True)
# item_detail_original.columns = header

In [8]:
# item_group_original = pd.read_csv(r"D:\IT\Datasets\MOT\item_group.csv",header=None, names=range(3))
# first_col = item_group_original[0].str.split('|', expand=True)
# header = first_col.iloc[0]
# item_group_original = first_col.iloc[1:].reset_index(drop=True)
# item_group_original.columns = header

It's better to make copies to ease rollbacks.

In [9]:
tr = test_result_original.copy()
ti = test_items_original.copy()
itde = item_detail_original.copy()
ig = item_group_original.copy()

# 2. Initial Data Exploration and Cleaning

## 'Test Result' table

### Drop columns, duplicates and change dtypes

In [10]:
tr.head()

,Unnamed: 0,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
0,0,1731451755,428817244,2024-01-22,1,NT,P,16319.0,W,YAMAHA,XC 125 E VITY,BLUE,PE,124.0,2010-08-21,2024-01-22T10:21:48.000Z
1,1,1165003385,771832290,2024-01-23,1,NT,P,33305.0,NN,HONDA,XL 125 V-4,BLUE,PE,125.0,2004-08-19,2024-01-23T14:03:20.000Z
2,2,635761943,43435464,2024-01-18,2,NT,P,16478.0,B,HONDA,CBR 600 FA-B,WHITE,PE,599.0,2011-12-22,2024-01-18T13:36:28.000Z
3,3,1585139025,1288564542,2024-01-25,2,NT,F,4273.0,SL,PIAGGIO,VESPA,CREAM,PE,200.0,2010-03-10,2024-01-25T11:44:25.000Z
4,4,272083495,1288564542,2024-01-25,2,RT,P,4273.0,SL,PIAGGIO,VESPA,CREAM,PE,200.0,2010-03-10,2024-01-25T17:10:14.000Z


In [11]:
buffer = io.StringIO()
tr.info(buf=buffer)
tr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391115 entries, 0 to 1391114
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1391115 non-null  int64  
 1   test_id            1391115 non-null  int64  
 2   vehicle_id         1391115 non-null  int64  
 3   test_date          1391115 non-null  object 
 4   test_class_id      1391115 non-null  int64  
 5   test_type          1391115 non-null  object 
 6   test_result        1391115 non-null  object 
 7   test_mileage       1362555 non-null  float64
 8   postcode_area      1391115 non-null  object 
 9   make               1391115 non-null  object 
 10  model              1391111 non-null  object 
 11  colour             1391115 non-null  object 
 12  fuel_type          1391115 non-null  object 
 13  cylinder_capacity  1384714 non-null  float64
 14  first_use_date     1391115 non-null  object 
 15  completed_date     1391115 non-n

First, we drop several unnecessary columns. The `['test_class_id']` column is removed because the dataset contains only motorcycles. We remove `['fuel_type']` because 99.87% are type "Petrol" and `['colour']` column.

In [12]:
tr_unnamed_column = tr.iloc[:,0].name
tr = tr.drop(columns=[tr_unnamed_column, 'test_class_id', 'colour', 'fuel_type'])

In [13]:
display(
    Markdown(
        f"Unfortunately, {tr.duplicated().sum()} of {tr.shape[0]} rows are duplicates, "
        f"so they are removed. As we have unique test id for each test we recheck duplicates again."
    )
)
tr = tr.drop_duplicates(keep='last')

Unfortunately, 410547 of 1391115 rows are duplicates, so they are removed. As we have unique test id for each test we recheck duplicates again.

In [14]:
display(
    Markdown(
        f"Still we have {tr.duplicated(subset='test_id').sum()} duplicates. Check what columns have most duplicated values."
    )
)

Still we have 49167 duplicates. Check what columns have most duplicated values.

In [15]:
test_id_dupl = tr[tr.duplicated(subset=['test_id'], keep=False)]
nunique_test_ids = test_id_dupl.groupby('test_id').nunique()
nunique_test_ids.gt(1).sum().sort_values(ascending=False)

first_use_date       46470
model                 1606
test_mileage           758
cylinder_capacity      214
vehicle_id             148
postcode_area          112
make                    68
test_date                0
test_type                0
test_result              0
completed_date           0
dtype: int64

As we don't have duplicates for last 4 columns `(['test_date', 'test_type', 'test_result', 'completed_date'])` we can suppose there were some mistakes with filling columns, the tests are distinct. So we drop duplicates again.

In [16]:
tr = tr.drop_duplicates(subset=['test_id'], keep='last')

Then we assign data types to the columns. ID columns are converted to the "Int32" type.
We remove the timezone from `['completed_date']`, since the dataset only contains data from the UK.
`['test_type']` and `['test_result']` are converted to the "category" type because they contain only 4 and 6 unique values respectively.

In [17]:
date_cols = ['test_date', 'first_use_date', 'completed_date']
for col in date_cols:
    tr[col] = pd.to_datetime(tr[col], errors='coerce')
tr['completed_date'] = tr['completed_date'].dt.tz_localize(None)

int_cols = ['test_mileage', 'cylinder_capacity', 'test_id', 'vehicle_id']
tr[int_cols] = tr[int_cols].astype('Int32')

cat_cols = ['test_type', 'test_result']
tr[cat_cols] = tr[cat_cols].astype('category')

str_cols = tr.select_dtypes(exclude=['Int32', 'datetime', 'category']).columns
tr[str_cols] = tr[str_cols].astype('string')

### Cleaning

#### ['make'] column

In [18]:
brands_lt5 = (tr['make'].value_counts() < 5).sum()
display(
    Markdown(
        f"We have {brands_lt5} of {tr['make'].nunique()} brands with less than 5 motorcycles. After column exploration "
        f"we see that values include not only brand name but model."
    )
)
print(tr.loc[tr['make'].str.contains('HON'), 'make'].value_counts()[10:20])

We have 1451 of 1836 brands with less than 5 motorcycles. After column exploration we see that values include not only brand name but model.

make
HONDA VFR400                            3
HONGYANG HY250-3                        3
HONDA MST260GED                         2
ZHEJIANG HONGZHOU                       2
DIRECT BIKES PYTHON                     2
LIFAN HONGDA LF125                      2
TAIZHOLM ZHONGNENG MOTORCYCLE CO LTD    2
ZHONGYU                                 2
HONDA NX650                             1
HONDA CB400SF-R                         1
Name: count, dtype: Int64


Some cars accidentally appear in the dataset, remove them.

In [19]:
tr = tr.drop(tr.loc[tr['make'].isin(['LOTUS', 'LAND ROVER'])].index)

We have 128 incorrect spelling for Harley-Davidson along with make & model together for some rows. So we edit wrong brand name spelling and convert it to one value and drop model name.

In [20]:
harley = tr.loc[tr['make'].str.contains('HARL')]
print(f"Unique instances of Harley-Davidson spelling: {harley['make'].nunique()}")
print('')
print('Model spelling value count:')
print(harley['make'].value_counts().head(10))

Unique instances of Harley-Davidson spelling: 129

Model spelling value count:
make
HARLEY-DAVIDSON                               28560
HARLEY DAVIDSON                               20748
HARLEY-DAVIDSON IRON 883 XL N 13                 69
HARLEY-DAVIDSON FLSTF FATBOY 1690 12             44
HARLEY-DAVIDSON FAT BOB FXDF 103 1690 13         38
HARLEYDAVIDSON                                   33
HARLEY-DAVIDSON STREET BOB FXDB 1584 13          31
HARLEY-DAVIDSON FORTY EIGHT XL 1200 X 13         29
HARLEY-DAVIDSON HERITAGE STC FLSTC 1690 13       29
HARLEY-DAVIDSON XL 883 N IRON 12                 28
Name: count, dtype: Int64


In [21]:
harley['make_copy'] = harley['make'].str.extract(r'^(HARLE?Y\s?-?DAVID?SON)\s?(.*)$')[0]
harley['make'] = 'HARLEY-DAVIDSON'
harley.drop(columns=['make_copy'], inplace=True)
tr.loc[harley.index] = harley

According to Wikipedia, Indian and Indian Motorcycle refer to one brand, so the names are unified.

In [22]:
print(tr.loc[tr['make'].str.contains('INDIAN'), 'make'].value_counts())
tr.loc[tr['make'].str.contains('INDIAN'), 'make'] = 'INDIAN MOTORCYCLE'

make
INDIAN MOTORCYCLE     2080
INDIAN                 590
INDIAN POLARIS           2
INDIAN MOTORCYCLES       1
INDIAN CHIEF             1
INDIAN SCOUT 60          1
Name: count, dtype: Int64


#### ['model'] column

The state of the column is that we have not only discrete name of the model for particular brand, but names of the series like CBR for `['make']` Honda. As an example, this model series has motorcycles with `['cylinder_capacity']` from 125 to 1000, meaning it includes various types and there is no way to distinct it.

In [23]:
print(tr.loc[(tr['make']=='HONDA') & (tr['model']=='CBR'), 'cylinder_capacity'].value_counts().head(10))
honda_models_num = tr.loc[(tr['make']=='HONDA'), 'model'].nunique()
honda_model_with_one_row = (tr.loc[(tr['make']=='HONDA'), 'model'].value_counts() == 1).sum()

cylinder_capacity
125     2213
649     1803
1000    1325
471      928
599      527
998      470
286      210
999      109
600       94
250       41
Name: count, dtype: Int64


In [24]:
display(
    Markdown(f'At the same time, the column has discrete model names like CBR600F which '
             f'is possibly a part of CBR series. And the most important thing, out '
             f'of {honda_models_num} Honda models, {honda_model_with_one_row} have only one row ' 
             f'so this column is quite hard to analyze therefore to save a memory we drop it '
             f' later but save vehicles IDs for two popular brands with the same cylinder capacity to analyze afterwards.'
    )
)

At the same time, the column has discrete model names like CBR600F which is possibly a part of CBR series. And the most important thing, out of 4029 Honda models, 1933 have only one row so this column is quite hard to analyze therefore to save a memory we drop it  later but save vehicles IDs for two popular brands with the same cylinder capacity to analyze afterwards.

In [25]:
honda_vs_yamaha = tr.loc[tr['model'].isin(['FZS600', 'CBR600F']), 'vehicle_id'].unique()

#### ['cylinder_capacity'] column

In [26]:
cc = tr['cylinder_capacity'].dropna()

cc_q1 = cc.quantile(0.25)
cc_median = cc.quantile(0.50)
cc_q3 = cc.quantile(0.75)
cc_iqr = cc_q3 - cc_q1

cc_lower_fence = max(cc.min(), cc_q1 - 1.5 * cc_iqr)
cc_upper_fence = min(cc.max(), cc_q3 + 1.5 * cc_iqr)

cc_outliers = cc[
    (cc < cc_lower_fence) |
    (cc > cc_upper_fence)
]

fig = go.Figure()
fig.add_trace(
    go.Box(
        q1=[cc_q1],
        median=[cc_median],
        q3=[cc_q3],
        lowerfence=[cc_lower_fence],
        upperfence=[cc_upper_fence],
        x=[''],
        boxpoints=False,
        name=''
    )
)

fig.add_trace(
    go.Scatter(
        x=[''] * len(cc_outliers),
        y=cc_outliers,
        mode='markers',
        marker=dict(size=5),
        name='Outliers',
        hovertemplate='Cylinder capacity: %{y}<extra></extra>'
    )
)

fig.update_layout(
    title='Cylinder capacity spread',
    xaxis=dict(
        title='',
        showticklabels=False
    ),
    yaxis=dict(
        title='',
        tickformat=',.0f'
    ),
    width=400,
    height=450,
    showlegend=False
)

fig.show()

The boxplot is used to check the spread of the column, it indicates that `['cylinder_capacity']` (cc) values above 2127 are outliers. Let's dive deeper. A simple fact check reveals that one of the highest cc for motorcycle is 2458 (Triumph Rocket 3) so we flag any value above 2500 and replace "99999" with "np.nan".


In [27]:
tr.loc[tr['cylinder_capacity']==99999, 'cylinder_capacity'] = np.nan

In [28]:
cc_outliers_upperbound = tr.loc[tr['cylinder_capacity']>2500].sort_values(by='cylinder_capacity', ascending=False)
cc_outliers_upperbound.groupby('cylinder_capacity').size().sort_index(ascending=False).head(10)

cylinder_capacity
13585     1
12999     1
12009     1
9976     17
9967      1
9923      1
9488      3
8897      1
8300      1
8200      1
dtype: int64

After a brief investigation, we found a discrepancy in the `[cylinder_capacity]` values. A dot was missing for many values except for Boss Hoss, it has highest cc values in table (5700, 8200), thereafter we will divide other values by 10.

In [29]:
cc_outliers_upperbound = cc_outliers_upperbound.loc[~cc_outliers_upperbound['make'].str.contains('BOSS|V8', regex=True)]

cc_outliers_upperbound['cylinder_capacity'] = (cc_outliers_upperbound['cylinder_capacity'] / 10).round(0)
tr.loc[cc_outliers_upperbound.index, 'cylinder_capacity'] = cc_outliers_upperbound['cylinder_capacity']

Since the lower bound for motorcycles is around 50 cc, values below 40 cc are removed.

In [30]:
tr.loc[tr['cylinder_capacity'] < 40, 'cylinder_capacity'] = np.nan

#### ['test_result'] column

If `['test_result']` is "P"(Passed) or "PRS"(Passed with rectification) it means the vehicle passed the test and won't appear in the table again. Let's check this assumption.

In [31]:
tr = tr.sort_values(by=['vehicle_id', 'completed_date'])
appear_after_pass = pd.DataFrame()
for i in range(4):
    test = tr.groupby('vehicle_id').nth(i)
    if test.empty:
        break
    passed_test = test.loc[lambda x: x['test_result'].isin(['P','PRS']), 'vehicle_id'].values
    passed_but_have_second = (
        tr.loc[tr['vehicle_id'].isin(passed_test)]
        .groupby('vehicle_id', as_index=False)
        .size()
        .sort_values(by='size', ascending=False)
        .loc[lambda x:x['size']>i+1]
    )
    print(f'{passed_but_have_second.shape[0]} vehicles passed {i+1}th test but have next one')
    appear_after_pass = pd.concat([appear_after_pass, passed_but_have_second])

10001 vehicles passed 1th test but have next one
759 vehicles passed 2th test but have next one
24 vehicles passed 3th test but have next one
2 vehicles passed 4th test but have next one


In [32]:
tr = tr.loc[~tr['vehicle_id'].isin(appear_after_pass['vehicle_id'])]

According to test guide, if certain motorcycle part fails the test(has result "F") in categories "Major" or "Dangerous" of `itde['rfr_deficiency_category']` column, then test is considered as failed. Later in analysis, after `tr(Test Result)` table is joined with other tables to obtain test result details, we have few motos, which have "P(Passed)" in `['test_result']` column, though have "F(Failed)" in `itde['rfr_deficiency_category']` column. It's better to delete it.

In [33]:
tr = tr.drop(tr.loc[tr['vehicle_id'].isin([278099932, 323120436, 695051878, 28929412])].index)

#### ['test_type'] column

According to the test guide, first test for each vehicle is labeled as "NT(Full initial test)", if it's second, third and so on, it's "RT(Re-Test)". 

In [34]:
tr['test_type'].value_counts()

test_type
NT    849398
RT     59350
EI         0
ES         0
Name: count, dtype: int64

In [35]:
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    837204
RT       795
EI         0
ES         0
Name: count, dtype: int64

In some cases, the first test is labeled as "RT" (Re-Test). It is unclear whether the previous test failed or took place in 2023, so to avoid discrepancy these tests will be dropped.

In [36]:
tr = tr.loc[~tr['vehicle_id'].isin(first_test.loc[lambda x: x['test_type']=='RT', 'vehicle_id'])]
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    837204
EI         0
ES         0
RT         0
Name: count, dtype: int64

Check other tests

In [37]:
other_test = tr.loc[~tr['test_id'].isin(first_test['test_id'])]
other_test['test_type'].value_counts()

test_type
RT    58553
NT    12189
EI        0
ES        0
Name: count, dtype: int64

We have similar situation for some tests that have type "NT", but their order isn't 1st. They will be dropped as well.

In [38]:
tr = tr.loc[~tr['vehicle_id'].isin(other_test.loc[lambda x: x['test_type']=='NT', 'vehicle_id'])]
first_test = tr.sort_values(['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(0).reset_index(drop=True)
first_test['test_type'].value_counts()

test_type
NT    825346
EI         0
ES         0
RT         0
Name: count, dtype: int64

#### ['first_use_date'] column

According to MOT test guide, vehicles that belong to 1st and 2nd `['test_class_id']` have first test in 3 years after first use, so maximum value for `['first_use_date']` column should be 2021. 

In [39]:
less_than_three_years_vehicles = tr.loc[tr['first_use_date'].dt.year > 2021, 'vehicle_id']
display(Markdown(f'We have {less_than_three_years_vehicles.shape[0]} motos, which were made in 2021 and later, so we drop it.'))
tr = tr.loc[~tr['vehicle_id'].isin(less_than_three_years_vehicles)]

We have 3649 motos, which were made in 2021 and later, so we drop it.

#### Other columns

In [40]:
display(
    Markdown(
        f"As we have only {(tr['test_date'].dt.day != tr['completed_date'].dt.day).sum()} tests "
       f"that weren't completed within a day we have no need in `['test_date']` column."
    )
)
tr = tr.drop('test_date', axis=1)

As we have only 6 tests that weren't completed within a day we have no need in `['test_date']` column.

There are some motorcycles that were produced before 1920, so we check whether the corresponding brands were founded earlier. 

In [41]:
tr.loc[tr.index.isin(tr['first_use_date'].sort_values().index[:10])].sort_values(by='first_use_date')[['make', 'first_use_date']]

,make,first_use_date
393909,LAMBRETTA,1866-08-01
796892,HARLEY-DAVIDSON,1900-01-01
945178,THUMPSTAR,1901-01-01
1081727,SUZUKI,1901-01-01
385893,HARLEY-DAVIDSON,1901-01-01
1295864,QUADRANT,1903-12-31
1233971,MINERVA,1904-12-31
1352725,INDIAN MOTORCYCLE,1914-12-31
901773,IXION,1921-01-14
1047803,RUDGE WHITWORTH,1921-01-24


Harley-Davidson, Lambretta, Suzuki, Thumpstar were founded after 1920, thus we change the dates to "pd.NaT", other brands were founded before indicated date of first use.

In [42]:
tr.loc[
    (tr['first_use_date'] < '1920-01-01') 
    & (tr['make'].isin(['HARLEY-DAVIDSON', 'LAMBRETTA', 'THUMPSTAR', 'SUZUKI']))
] = np.nan

`['Age']` column is created for further analysis.

In [43]:
tr['age'] = ((tr['completed_date'] - tr['first_use_date']).dt.days / 365.25).round(2)

To check `['test_mileage']` column, we add column `['miles_per_year']` to drop outliers.

In [44]:
tr['miles_per_year'] = (tr['test_mileage'] / tr['age']).round(2)
display(Markdown(f'Top 10 test mileage:'))
(tr.loc[
    tr.index.isin(
        tr['test_mileage'].sort_values(ascending=False)
        .index[:10]
        )
    ].sort_values(by='miles_per_year', ascending=False)
    [['first_use_date', 'test_mileage', 'miles_per_year']]
)

Top 10 test mileage:

,first_use_date,test_mileage,miles_per_year
730741,2018-09-07,884216,152978.55
470719,2017-05-20,704364,101056.53
291059,2013-08-01,949647,88918.26
124673,2006-03-14,999999,55679.23
1211538,2012-05-01,657982,52849.96
342587,2005-03-09,999999,52383.39
517024,2001-12-31,929980,41609.84
517026,2001-12-31,929990,41591.68
1176533,1991-02-18,820702,24411.12
1188405,1994-07-23,626639,20790.94


For the upper bound, the value "999999" in `['test_mileage']` seems suspicious and may be a placeholder for missing values. Therefore, it is replaced with "np.nan". Similarly, values higher than 60000 for `['miles_per_year']` are replaced with "np.nan", since it is unrealistic to cover, for instance, 243 miles per day (88938 / 365) for many years on a motorcycle.

In [45]:
tr.loc[(tr['test_mileage'] == 999999) | (tr['miles_per_year'] >= 60000), 'test_mileage'] = np.nan
tr = tr.drop(columns=['miles_per_year'])

Simplified version of `['test_result']` column is added for further analysis.

In [46]:
tr['passed'] = tr['test_result'].isin(['P', 'PRS'])

In [47]:
tr['test_result'].value_counts()

test_result
P        769122
F         62539
PRS       46738
ABR         371
ABA          41
ABRVE         0
Name: count, dtype: int64

In [48]:
display(Markdown(f"Initial table info:"))
print(buffer.getvalue())

Initial table info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391115 entries, 0 to 1391114
Data columns (total 16 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Unnamed: 0         1391115 non-null  int64  
 1   test_id            1391115 non-null  int64  
 2   vehicle_id         1391115 non-null  int64  
 3   test_date          1391115 non-null  object 
 4   test_class_id      1391115 non-null  int64  
 5   test_type          1391115 non-null  object 
 6   test_result        1391115 non-null  object 
 7   test_mileage       1362555 non-null  float64
 8   postcode_area      1391115 non-null  object 
 9   make               1391115 non-null  object 
 10  model              1391111 non-null  object 
 11  colour             1391115 non-null  object 
 12  fuel_type          1391115 non-null  object 
 13  cylinder_capacity  1384714 non-null  float64
 14  first_use_date     1391115 non-null  object 
 15  completed_date     1391115 non-n

In [49]:
display(Markdown(f"Table info after tranformation:"))
tr.info()

Table info after tranformation:

<class 'pandas.core.frame.DataFrame'>
Index: 878816 entries, 811434 to 1017787
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   test_id            878811 non-null  Int32         
 1   vehicle_id         878811 non-null  Int32         
 2   test_type          878811 non-null  category      
 3   test_result        878811 non-null  category      
 4   test_mileage       867017 non-null  Int32         
 5   postcode_area      878811 non-null  string        
 6   make               878811 non-null  string        
 7   model              878810 non-null  string        
 8   cylinder_capacity  875186 non-null  Int32         
 9   first_use_date     878809 non-null  datetime64[ns]
 10  completed_date     878811 non-null  datetime64[ns]
 11  age                878809 non-null  float64       
 12  passed             878816 non-null  bool          
dtypes: Int32(4), bool(1), category(2), datetime

## 'Item Detail' table

In [50]:
itde.head()

,rfr_id,test_class_id,test_item_id,minor_item,rfr_deficiency_category,rfr_desc,rfr_loc_marker,rfr_insp_manual_desc,rfr_advisory_text,test_item_set_section_id
0,4,1,3,Y,Pre-EU Directive,missing,N,an obligatory lamp missing,NaN,1
1,4,2,3,Y,Pre-EU Directive,missing,N,an obligatory lamp missing,NaN,1
2,5,1,3,Y,Pre-EU Directive,damaged and function impaired,N,so damaged or deteriorated that its function i...,damaged but function not impaired,1
3,5,2,3,Y,Pre-EU Directive,damaged and function impaired,N,so damaged or deteriorated that its function i...,damaged but function not impaired,1
4,6,1,3,Y,Pre-EU Directive,deteriorated and function impaired,N,so damaged or deteriorated that its function i...,deteriorated but function not impaired,1


In [51]:
itde.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21069 entries, 0 to 21068
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   rfr_id                    21069 non-null  int64 
 1   test_class_id             21069 non-null  int64 
 2   test_item_id              21069 non-null  int64 
 3   minor_item                21069 non-null  object
 4   rfr_deficiency_category   21069 non-null  object
 5   rfr_desc                  21069 non-null  object
 6   rfr_loc_marker            21069 non-null  object
 7   rfr_insp_manual_desc      20911 non-null  object
 8   rfr_advisory_text         6824 non-null   object
 9   test_item_set_section_id  21069 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 1.6+ MB


In [52]:
buffer = io.StringIO()
itde.info(buf=buffer)

Here we won't filter `['test_class_id']` column, because due to discrepancy we'll have NaN for some rows later if we drop it. 

In [53]:
# # original 
# itde = itde.loc[itde['test_class_id'].isin([1,2])]
# itde = itde.drop(columns=['test_class_id', 'rfr_desc','rfr_advisory_text', 'rfr_loc_marker'])
# itde.rename(columns={'rfr_insp_manual_desc': 'rfr_desc'}, inplace=True)
# itde['minor_item'] = itde['minor_item'].map({'Y': True, 'N': False})

In [54]:
itde['rfr_insp_manual_desc'] = itde['rfr_insp_manual_desc'].fillna(itde['rfr_desc'])
itde = itde.drop(columns=['test_class_id','rfr_desc','rfr_advisory_text', 'rfr_loc_marker'])
itde.rename(columns={'rfr_insp_manual_desc': 'rfr_desc'}, inplace=True)
itde['minor_item'] = itde['minor_item'].map({'Y': True, 'N': False})

Again, i'll convert all ids column to "Int32" type and columns with low number of unique values to "category".

In [55]:
int_cols = ['rfr_id', 'test_item_id', 'test_item_set_section_id']
itde[int_cols] = itde[int_cols].astype('Int32')
itde['rfr_deficiency_category'] = itde['rfr_deficiency_category'].astype('category')
itde['rfr_desc'] = itde['rfr_desc'].astype('string')

In [56]:
itde = itde.drop_duplicates(keep='first')

In [57]:
display(Markdown(f"Initial table info:"))
print(buffer.getvalue())

Initial table info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21069 entries, 0 to 21068
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   rfr_id                    21069 non-null  int64 
 1   test_class_id             21069 non-null  int64 
 2   test_item_id              21069 non-null  int64 
 3   minor_item                21069 non-null  object
 4   rfr_deficiency_category   21069 non-null  object
 5   rfr_desc                  21069 non-null  object
 6   rfr_loc_marker            21069 non-null  object
 7   rfr_insp_manual_desc      20911 non-null  object
 8   rfr_advisory_text         6824 non-null   object
 9   test_item_set_section_id  21069 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 1.6+ MB



In [58]:
display(Markdown(f"Table info after tranformation:"))
itde.info()

Table info after tranformation:

<class 'pandas.core.frame.DataFrame'>
Index: 7505 entries, 0 to 21067
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   rfr_id                    7505 non-null   Int32   
 1   test_item_id              7505 non-null   Int32   
 2   minor_item                7505 non-null   bool    
 3   rfr_deficiency_category   7505 non-null   category
 4   rfr_desc                  7505 non-null   string  
 5   test_item_set_section_id  7505 non-null   Int32   
dtypes: Int32(3), bool(1), category(1), string(1)
memory usage: 242.1 KB


## 'Test Item' table

In [59]:
ti.head()

,Unnamed: 0,test_id,rfr_id,rfr_type_code,mot_test_rfr_location_type_id,dangerous_mark,completed_date
0,0,939637859,40098,A,9,NaN,2024-01-02T10:03:29.000Z
1,1,939637859,40098,A,9,NaN,2024-01-02T10:03:29.000Z
2,2,401677457,40450,A,25,NaN,2024-01-02T10:47:37.000Z
3,3,401677457,40546,A,83,NaN,2024-01-02T10:47:37.000Z
4,4,401677457,40663,A,5,NaN,2024-01-02T10:47:37.000Z


In [60]:
buffer = io.StringIO()
ti.info(buf=buffer)
ti.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815457 entries, 0 to 815456
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   Unnamed: 0                     815457 non-null  int64 
 1   test_id                        815457 non-null  int64 
 2   rfr_id                         815457 non-null  int64 
 3   rfr_type_code                  815457 non-null  object
 4   mot_test_rfr_location_type_id  815457 non-null  int64 
 5   dangerous_mark                 57545 non-null   object
 6   completed_date                 815457 non-null  object
dtypes: int64(4), object(3)
memory usage: 43.6+ MB


Useless columns and duplicates are dropped because each test for vehicle has its own ID.

In [61]:
ti_unnamed_column = ti.iloc[:,0].name
ti.drop(columns=[ti_unnamed_column, 'mot_test_rfr_location_type_id'], inplace=True)
ti = ti.drop_duplicates(keep='first').reset_index(drop=True)

As we have "P(Passed)" value in `tr['test_result']` we change to "P" to "PRS(Pass with rectification)" in `ti['rfr_type_code']`

In [62]:
ti['rfr_type_code'] = ti['rfr_type_code'].str.replace('P', 'PRS')

Data types

In [63]:
ti['completed_date'] = pd.to_datetime(ti['completed_date'])
ti['completed_date']  = ti['completed_date'].dt.tz_localize(None)

int_cols = ['test_id', 'rfr_id']
ti[int_cols] = ti[int_cols].astype('Int32')
ti['rfr_type_code'] = ti['rfr_type_code'].astype('category')

#### ['Dangerous_mark'] column

MOT test data user guide describes this column as a field where a tester could choose to denote that a failure item was dangerous, but it isn't directly comparable to the "Dangerous" category brought in with the EU Roadworthiness Directive from 20th May 2018. So to avoid confusion it is dropped.

In [64]:
ti = ti.drop(columns=['dangerous_mark'])

In [65]:
display(Markdown(f"Initial table info"))
print(buffer.getvalue())

Initial table info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815457 entries, 0 to 815456
Data columns (total 7 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   Unnamed: 0                     815457 non-null  int64 
 1   test_id                        815457 non-null  int64 
 2   rfr_id                         815457 non-null  int64 
 3   rfr_type_code                  815457 non-null  object
 4   mot_test_rfr_location_type_id  815457 non-null  int64 
 5   dangerous_mark                 57545 non-null   object
 6   completed_date                 815457 non-null  object
dtypes: int64(4), object(3)
memory usage: 43.6+ MB



In [66]:
display(Markdown(f"Table info after tranformation"))
ti.info()

Table info after tranformation

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649389 entries, 0 to 649388
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   test_id         649389 non-null  Int32         
 1   rfr_id          649389 non-null  Int32         
 2   rfr_type_code   649389 non-null  category      
 3   completed_date  649389 non-null  datetime64[ns]
dtypes: Int32(2), category(1), datetime64[ns](1)
memory usage: 11.8 MB


## 'Item group' table

In [67]:
ig.head()

,test_item_id,test_class_id,parent_id,test_item_set_section_id,item_name
0,0,1,0,0,Vehicle
1,0,2,0,0,Vehicle
2,0,3,0,0,Vehicle
3,0,4,0,0,Vehicle
4,0,5,0,0,Vehicle


In [68]:
ig = ig.loc[ig['test_class_id'].isin([1,2])].drop(columns=['test_class_id'])
int_cols = ['test_item_id', 'parent_id', 'test_item_set_section_id']
ig[int_cols] = ig[int_cols].astype('Int32')

str_cols = ig.select_dtypes(exclude='int').columns
ig[str_cols] = ig[str_cols].astype('string')

In [69]:
ig.info()

<class 'pandas.core.frame.DataFrame'>
Index: 632 entries, 0 to 4135
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   test_item_id              632 non-null    Int32 
 1   parent_id                 632 non-null    Int32 
 2   test_item_set_section_id  632 non-null    Int32 
 3   item_name                 632 non-null    string
dtypes: Int32(3), string(1)
memory usage: 19.1 KB


# 3. Preliminary Analysis (Test Result)

## Table  summary

As it was mentioned before, `['make']` column has many rows where model is included so it's hard to count number of brands here.

In [70]:
display(Markdown(f"Total number of motorcycles: {tr['vehicle_id'].nunique()}"))
display(Markdown(f"Number of brands with 5 or more tests: {(tr['make'].value_counts() >= 5).sum()}"))

Total number of motorcycles: 821884

Number of brands with 5 or more tests: 319

In [71]:
display(Markdown(f"Number of motorcycles passed the test by number of attempts:"))
moto_num_by_test_num = (
    tr.groupby('vehicle_id', as_index=False)
    .size()
    .rename(columns={'size':'attempts_num'})['attempts_num']
    .value_counts()
    .to_frame('number_of_moto')
    .reset_index()
)
moto_num_by_test_num

Number of motorcycles passed the test by number of attempts:

,attempts_num,number_of_moto
0,1,765112
1,2,56618
2,3,153
3,4,1


In [72]:
display(Markdown("Test results:"))
for test_num in range(4):
    test_result = tr.sort_values(by=['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(test_num)['passed'].value_counts()
    passed = test_result.get(True, 0)
    failed = test_result.get(False, 0)
    fail_pct = round(failed / (failed + passed)* 100, 2)
    abandoned = 0
    if test_num > 0:
        pr_test_result = tr.sort_values(by=['vehicle_id', 'completed_date']).groupby('vehicle_id').nth(test_num-1)['passed'].value_counts()
        pr_test_failed = pr_test_result.get(False, 0)
        abandoned = pr_test_failed - (passed + failed)
    fail_pct_str = f'{fail_pct:.2f}%'
    print(f'Test #{test_num+1}:')
    print(f'{"Passed:":<10}{passed:>8}')
    print(f'{"Failed:":<10}{failed:>8}')
    print(f'{"Fail rate:":<10}{fail_pct_str:>8}')
    print(f'{"No retry:":<10}{abandoned:>8}')
    print('')

Test results:

Test #1:
Passed:     759142
Failed:      62742
Fail rate:   7.63%
No retry:        0

Test #2:
Passed:      56564
Failed:        208
Fail rate:   0.37%
No retry:     5970

Test #3:
Passed:        153
Failed:          1
Fail rate:   0.65%
No retry:       54

Test #4:
Passed:          1
Failed:          0
Fail rate:   0.00%
No retry:        0



Interestingly, some motorcycles did not retake the test. Later in analysis we dive deeper for reasons of it.

In [73]:
first_test = tr.loc[tr['test_type']=='NT'].drop(columns=['test_type']).reset_index(drop=True)
passed_without_flaws = (~first_test.loc[first_test['test_result']=='P', 'test_id'].isin(ti['test_id'])).sum()

display(
    Markdown(
        f"We don't have any data in `Test Item` table for {passed_without_flaws} of {tr['test_result'].value_counts()['P']} motos which passed the test,"
        f" meaning they passed it without any flaws."
    )
)

We don't have any data in `Test Item` table for 539993 of 769122 motos which passed the test, meaning they passed it without any flaws.

In [74]:
test_result_cnt = tr['test_result'].value_counts()
test_result_cnt

test_result
P        769122
F         62539
PRS       46738
ABR         371
ABA          41
ABRVE         0
Name: count, dtype: int64

In [75]:
display(
    Markdown(
        f"Out of all results we have {test_result_cnt['ABR']} tests that were aborted(i.e test cannot be "
        f"completed because of a problem with the testing equipment or the NT). We check what finally happened to them."
    )
)

Out of all results we have 371 tests that were aborted(i.e test cannot be completed because of a problem with the testing equipment or the NT). We check what finally happened to them.

In [76]:
abrt_moto_vals = tr.loc[tr['test_result']=='ABR', 'vehicle_id'].values
abrt_moto = tr.loc[tr['vehicle_id'].isin(abrt_moto_vals)]
abrt_moto_last_test = abrt_moto.sort_values(by=['vehicle_id', 'completed_date']).groupby('vehicle_id').last()
abrt_moto_cnt = abrt_moto_last_test['test_result'].value_counts()

display(
    Markdown(
        f"We have {abrt_moto_cnt['ABR']} motos whose test was aborted and they didn't retake it. "
        f"How much time on average it takes for vehicle owners to retake the test?"
    )
)

We have 218 motos whose test was aborted and they didn't retake it. How much time on average it takes for vehicle owners to retake the test?

In [77]:
time_between_tests = (
    tr[tr.duplicated('vehicle_id', keep=False)]
        .sort_values(by=['vehicle_id', 'completed_date'])
        .groupby('vehicle_id')['completed_date'].diff()
        .dropna()
        .describe()
)
time_between_tests

count                        56927
mean     3 days 12:39:40.075025910
std      3 days 22:49:22.778871873
min                0 days 00:00:46
25%         0 days 03:38:00.500000
50%                1 days 23:57:02
75%         6 days 01:38:25.500000
max              216 days 04:55:26
Name: completed_date, dtype: object

There are 56927 retests, the average time between tests is about 3 hours 12 hours but this mean is skewed by extreme values such as 216 days. Therefore, the median (50th percentile) is more representative measure, which is approximately 2 days. Additionally, 75% of retests occur within 6 days and 1 hour.

### Free or partial fee retest
[Official goverment services website](https://www.gov.uk/getting-an-mot/retests) states there is either no fee for vehicle owners who need a partial retest if they leave the vehicle at the test centre for repair and it’s retested within 10 working days or partial fee if the owner takes the vehicle from the test centre for repairs and takes it back within 10 working days. However, the [website](https://www.gov.uk/getting-an-mot/after-the-test) also states that the owner may take the vehicle away if the current MOT is still valid, if dangerous defects aren't listed. Therefore we check how many vehicles qualify for free retest/discount. It means we count how many days passed after the test before retest.

In [78]:
# more_than_one_test = (
#     tr.loc[tr['vehicle_id'].isin(tr.groupby('vehicle_id').size()[tr.groupby('vehicle_id').size() > 1].index)]
#     .sort_values(by=['vehicle_id', 'completed_date'])
# )

# def busday_diff(date_list):
#     diff_list = []
#     for idx in range(len(date_list) - 1):
#         start = date_list[idx].to_numpy().astype('datetime64[D]')
#         end = date_list[idx+1].to_numpy().astype('datetime64[D]')
#         diff = np.busday_count(start, end, weekmask=[1,1,1,1,1,1,0])
#         diff_list.append(diff)
#     return diff_list
# days_num_between_tests = (
#     more_than_one_test.groupby('vehicle_id')['completed_date'].agg(list)
#         .apply(busday_diff)
#         .reset_index(name='days_num')
# )
# days_num_between_tests = days_num_between_tests.explode('days_num')
# paid_retest = days_num_between_tests.loc[days_num_between_tests['days_num'] > 10].shape[0]
# free_retest = days_num_between_tests.loc[days_num_between_tests['days_num'] <= 10].shape[0]

# display(Markdown(f"{free_retest} of {days_num_between_tests.shape[0]} motos got no fee or partial fee retest."))

### Number of tests by time periods

Now we would like to know how many tests were taken each month.

In [79]:
tests_by_month = (
    tr.groupby([tr['completed_date'].dt.month, tr['completed_date'].dt.strftime('%b')])
    .size()
    .droplevel(level=0)
)
tests_by_month = tests_by_month.reset_index(name='test_num').rename(columns={'completed_date': 'month'})

In [80]:
fig = px.line(
    tests_by_month, 
    x='month', 
    y='test_num', 
    title="Number of tests by month", 
    markers=True
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(title=''),
    yaxis = dict(
        title='',
        tickformat=',.0f'
    ),
    title=dict(y=0.9)
)
fig.show()

We see increasing number of tests from start of the year till May and sequential falling to the end of the year. What about day of the week?

In [81]:
tests_by_day = (
tr.groupby([tr['completed_date'].dt.dayofweek, tr['completed_date'].dt.day_name()])
    .agg(test_num=('completed_date', 'size'))
    .droplevel(level=0)
    .reset_index()
    .rename(columns={'completed_date': 'day'})
)

In [82]:
fig = px.bar(
    tests_by_day, 
    x='day', 
    y='test_num', 
    title="Number of tests by day of the week", 
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(title=''),
    yaxis=dict(
        title='',
        tickformat=',.0f',
    ),
    title=dict(y=0.9)
)
fig.show()

Test activity is relatively low on Mondays, comparably equal number of tests in the middle of week and its peak on Friday and significant drop on Saturday. Even small amount of tests took place on Sunday, possibly due to error in data or urgency on tests.
Next, we check the time of test completion.

In [83]:
completion_time = (
    tr['completed_date'].dropna().dt.hour
    .value_counts()
    .sort_index()
    .rename_axis('hour')
    .reset_index()
)
out_of_worktime_perc = (
    completion_time.loc[(completion_time['hour'] < 8) | (completion_time['hour'] > 17), 'count'].sum()
    / completion_time['count'].sum()
)
completion_time['hour'] = completion_time['hour'].astype(str).str.zfill(2)

In [84]:
fig = px.bar(
    completion_time, 
    y='count', 
    x='hour',  
    title="Number of tests by hour", 
    text_auto=True
)
fig.update_layout(
    width=1050,
    height=500,
    xaxis=dict(title=''),
    yaxis=dict(
        title='', 
        tickformat=',.0f',
    ),  
    title=dict(y=0.9),
)
fig.update_traces(
    textposition='outside', 
    textfont_size=10, 
    textfont_color='#6B7280',
    hoverinfo='skip', 
    hovertemplate=None
) 
fig.show()

In [85]:
non_working_time = completion_time.loc[(completion_time['hour'] <'07') | (completion_time['hour']>='17'), 'count'].sum()
оwh = (non_working_time / completion_time['count'].sum()).round(4)

display(
    Markdown(
        f"From this bar chart we can conclude most Vehicle Testing Stations (VTS) appear to operate between 07:00 and 17:00 and {оwh:.2%} "
        f"of all tests were completed after-hours or some values were entered by mistake (04:00 instead of 16:00)."
            )
        )

From this bar chart we can conclude most Vehicle Testing Stations (VTS) appear to operate between 07:00 and 17:00 and 2.08% of all tests were completed after-hours or some values were entered by mistake (04:00 instead of 16:00).

From this point onward, the analysis will be focused on first test only, because it is the most valuable source of data. Vehicle owners may not be aware of all defects before the first inspection.

### First test fail rate by month

Next, we examine whether the month of testing influences the results.

In [86]:
def fail_rate_calc(df, groupby_cols, test_num=None, sort_col=None, asc=True, to_drop=False):
    grouped_df = (
        df.groupby(groupby_cols, observed=False)['passed']
        .agg(
            fail_rate=lambda x: (x == False).sum() / len(x), 
             test_num='count')
    )
    if test_num is not None:
        grouped_df = grouped_df.loc[grouped_df['test_num'] >= test_num]
    if sort_col is not None:
        grouped_df = grouped_df.sort_values(by=sort_col, ascending=asc)
    grouped_df = grouped_df.reset_index(drop=to_drop)
    return grouped_df

In [87]:
month_num = first_test['completed_date'].dt.month.rename('month_num')
month_name = first_test['completed_date'].dt.strftime('%b').rename('month_name')

fail_rate_by_month = (
    first_test.groupby([month_num, month_name])
    ['passed']
    .agg(
        passed=lambda x:(x == True).sum(),
        failed=lambda x: (x == False).sum(),
        fail_rate=lambda x: (x == False).sum() / len(x),
        test_num='count'
    )
    .reset_index(level='month_name')
)  

In [88]:
fig = px.line(
    fail_rate_by_month, 
    y='fail_rate', 
    x='month_name',  
    title="Fail rate of first test by month",
    hover_data={'fail_rate': ':.2%'},
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(title=''),
    yaxis=dict(
        title='',
        tickformat='.1%',           
    ),
    title=dict(y=0.9)
)                 
fig.show()

Interestingly, the fail rate varies by month, and visually there appears to be a negative correlation between the number of tests and the fail rate by month.
To determine whether this pattern occurs by chance, we compute the chi-square statistic and p-value for a test of independence using the contingency table. The null hypothesis states that test results are independent of the month.

In [89]:
from scipy.stats import chi2_contingency

print('Month contigency table:')
print('')
print(fail_rate_by_month[['passed', 'failed']])
chi2, p_value, dof, expected = chi2_contingency(fail_rate_by_month[['passed', 'failed']])

print("\nChi-Square Test results:")
print(f"Chi2 Test Statistic: {chi2}")
print(f"P-value: {p_value}")
print(f"Degrees of Freedom (dof): {dof}")

alpha = 0.05
if p_value < alpha:
    print(f"Because p-value ({p_value:.4f}) < 0.05, we reject null hypothesis.")
    print("The difference in test fail rates between months is statistically significant.")
else:
    print(f"Because p-value ({p_value:.4f}) >= 0.05, we fail to reject null hypothesis.")
    print("We haven't enough evidence to say the difference is significant. It could be chance.")

Month contigency table:

           passed  failed
month_num                
1           23572    2453
2           46663    4181
3           80704    6398
4           83101    6334
5          116709    8596
6           94216    7031
7           84774    6893
8           71284    5906
9           59241    5118
10          52024    4851
11          29107    2979
12          17747    2002

Chi-Square Test results:
Chi2 Test Statistic: 742.202102702888
P-value: 4.808429482487106e-152
Degrees of Freedom (dof): 11
Because p-value (0.0000) < 0.05, we reject null hypothesis.
The difference in test fail rates between months is statistically significant.


Let's check the correlation between number of tests and it's fail rate.

In [90]:
fail_rate_by_month[['test_num', 'fail_rate']].corr()

,test_num,fail_rate
test_num,1.000000,-0.957492
fail_rate,-0.957492,1.000000


### Fail rate by region & postcode 

Next, we compare first test results across different regions, but first we need to group poscodes by regions, we will use a table from [www.robertsharp.co.uk](https://www.robertsharp.co.uk/2017/08/09/a-table-that-shows-the-uk-region-for-all-postcode-districts/). Non-geographic postcodes are excluded.

In [91]:
region_postcodes_dict = {
    "Scotland": ["AB", "DD", "DG", "EH", "FK", "G", "IV", "HS", "KA", "KW", "KY", "ML", "PA", "PH", "TD", "ZE"],
    "East of England": ["AL", "CB", "CM", "CO", "HP", "IP", "LU", "NR", "PE", "SG", "SS"],
    "West Midlands": ["B", "CV", "DY", "HR", "NN", "ST", "TF", "WR", "WS", "WV"],
    "South West": ["BA", "BH", "BS", "DT", "EX", "GL", "PL", "SN", "SP", "TA", "TQ", "TR"],
    "North West": ["BB", "BD", "BL", "CA", "CH", "CW", "FY", "HD", "HX", "L", "LA", "M", "OL", "PR", "SK", "WA", "WN"],
    "South East": ["BN", "CT", "GU", "ME", "MK", "OX", "PO", "RG", "RH", "SL", "SO", "TN"],
    "Greater London": ["BR", "CR", "DA", "E", "EC", "EN", "HA", "IG", "KT", "N", "NW", "RM", "SE", "SM", "SW", "TW", "UB", "W", "WC", "WD"],
    "Northern Ireland": ["BT"],
    "North East": ["DH", "DL", "HG", "HU", "LS", "NE", "SR", "TS", "WF", "YO"],
    "East Midlands": ["DE", "DN", "LE", "LN", "NG", "S"],
    "Wales": ["CF", "LD", "LL", "NP", "SA", "SY"],
    "Channel Islands": ["GY", "JE"],
    "Isle of Man": ["IM"],
}
postcode_region_dict = {
    postcode: region
    for region, postcodes in region_postcodes_dict.items()
    for postcode in postcodes
}

In [92]:
first_test_regions = first_test['postcode_area'].map(postcode_region_dict).rename('region')
test_num_by_region = first_test_regions.value_counts()

fail_rate_by_region = fail_rate_calc(
    first_test, 
    first_test_regions,
    test_num=None, 
    sort_col='fail_rate',
    asc=False,
    to_drop=False
)

In [93]:
fig = px.bar(
    fail_rate_by_region,
    x='region',
    y='fail_rate', 
    title="First test fail rate by region", 
    hover_data={
        "fail_rate": ':.2%',
        "test_num": True,
    },
)
fig.update_layout(
    width=1050,
    height=500,
    title=dict(y=0.9),
    xaxis=dict(
        title='',
        tickfont=dict(
            size=11
        )
    ),
    yaxis=dict(
        title='',
        tickformat='.0%'
    ),
)
fig.show()

We see south part of UK tends to have higher fail rate than north, however, Scotland that is situated northernmost, have lower fail rate than north of England. Then we dive deeper and put our high-level postcodes on map to get approximate area distribution where tests were taken place. Postcode area latitude and longitude data are downloaded from [Free Map Tools](https://www.freemaptools.com/).

In [94]:
uk_coordinates = pd.read_csv(r'https://data.freemaptools.com/download/uk-outcode-postcodes/postcode-outcodes.csv', index_col='id')
uk_coordinates['postcode'] = uk_coordinates['postcode'].str.extract(r'(^[A-Z]{1,2})')
centroids = uk_coordinates.groupby('postcode', as_index=False)[['latitude', 'longitude']].mean()
centroids.head()

,postcode,latitude,longitude
0,AB,57.264839,-2.440001
1,AL,51.777545,-0.271277
2,B,52.457823,-1.883388
3,BA,51.164461,-2.459054
4,BB,53.802076,-2.311302


To get postcode full name, i download table from [Wikipedia article](https://en.wikipedia.org/wiki/List_of_postcode_areas_in_the_United_Kingdom). It provides number of districts within area so it is used as well.

In [95]:
from io import StringIO
postcode_url = "https://en.wikipedia.org/wiki/List_of_postcode_areas_in_the_United_Kingdom"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
try:
    response = requests.get(postcode_url, headers=headers)
    response.raise_for_status()  
    html_content = response.text

    postcodes = pd.read_html(StringIO(html_content))[0]
    postcodes.columns = postcodes.columns.str.lower().str.replace(' ', '_')
    postcodes['districts_num_within_area'] = postcodes['number_of_postcode_districts_within_area'].str.split('(').str[0]
    postcodes = postcodes[['postcode_area', 'code_formation', 'districts_num_within_area']]
except requests.exceptions.HTTPError as err:
    print(f"HTTP error occurred: {err}")
except Exception as err:
    print(f"An error occurred: {err}")
postcodes.head()

,postcode_area,code_formation,districts_num_within_area
0,AB,Aberdeen,33
1,AL,St Albans,10
2,B,Birmingham,79
3,BA,Bath,19
4,BB,Blackburn,13


In [96]:
geo = (
    postcodes.merge(
        centroids, 
        left_on='postcode_area', 
        right_on='postcode', 
        how='left'
    )
    [['postcode_area', 'code_formation', 'latitude', 'longitude', 'districts_num_within_area']]
)

fail_rate_by_postcode = (
    first_test.groupby('postcode_area', as_index=False)
    .agg(
        test_num=('passed', 'size'),
        mean_age=('age', 'mean'),
        fail_rate=('passed', lambda x: (x== False).sum() / len(x))
    ).round(4)
)
fail_rate_by_postcode_geo = fail_rate_by_postcode.merge(geo, on='postcode_area')
fail_rate_by_postcode_geo['region'] = fail_rate_by_postcode_geo['postcode_area'].map(postcode_region_dict)
fail_rate_by_postcode_geo.head()

,postcode_area,test_num,mean_age,fail_rate,code_formation,latitude,longitude,districts_num_within_area,region
0,AB,6052,13.4912,0.0666,Aberdeen,57.264839,-2.440001,33,Scotland
1,AL,2829,12.6027,0.0520,St Albans,51.777545,-0.271277,10,East of England
2,B,14037,13.4861,0.0690,Birmingham,52.457823,-1.883388,79,West Midlands
3,BA,8531,14.1781,0.1172,Bath,51.164461,-2.459054,19,South West
4,BB,4810,13.6037,0.0572,Blackburn,53.802076,-2.311302,13,North West


In [97]:
fail_rate_by_postcode_geo[['mean_age', 'fail_rate']].corr()

,mean_age,fail_rate
mean_age,1.000000,-0.040525
fail_rate,-0.040525,1.000000


The correlation between mean age and fail rate for postcode areas is almost absent.

In [98]:
fig = px.scatter_map(
    fail_rate_by_postcode_geo,
    title='Fail rate by area',
    lat="latitude",
    lon="longitude",
    size="test_num",  
    color="fail_rate",
    color_continuous_scale="Agsunset",
    custom_data=['code_formation',
                 'region',
                 'postcode_area', 
                 'test_num', 
                 'districts_num_within_area', 
                 'mean_age', 
                 'fail_rate'
    ],
    map_style='open-street-map',
    zoom=5
)
fig.update_layout(
    width=600,
    height=600,
    title=dict(y=0.9),
    xaxis=dict(title=''),
    coloraxis_colorbar=dict(tickformat='.0%', title='fail rate'),
    coloraxis=dict(reversescale=True) 
)
fig.update_traces(
    hovertemplate=
        "<b>%{customdata[0]}</b><br>" +
        "Region: %{customdata[1]}<br>" +
        "Postcode area: %{customdata[2]}<br>" +
        "Number of tests: %{customdata[3]}<br>" +
        "Districts within area: %{customdata[4]}<br>" +
        "Mean age: %{customdata[5]:.2f}<br>" +
        "Fail rate: %{customdata[6]:.2%}<br>" 
        "<extra></extra>"
)

fig.show()

We see that rural and coastal areas on South-East and South-West tend to have higher fail rate on average. Some of code formations belonging to Greater London tends to have comparably low mean age but high fail rate.

### Number of motorycles by decade of registration

Now we'll visualize how many motorcycles were put into use in each decade, firstly we need to create bins.

In [99]:
decade_bins = pd.cut(
    first_test['first_use_date'].dt.year, 
    range(1920, 2031, 10), 
    right=False,
    labels=[
        "1920-1930", "1930-1940",
        "1940-1950","1950-1960",
        "1960-1970","1970-1980",
        "1980-1990","1990-2000", 
        "2000-2010", "2010-2020", 
        "2020-2021"
    ]
)
moto_num_by_decade = (
    decade_bins.value_counts()
    .reset_index(name='moto_num')
    .sort_values(by='first_use_date')
)

In [100]:
max_value = moto_num_by_decade['moto_num'].max()
buffer = max_value * 0.10  
x_axis_range = [0, max_value + buffer]

fig = px.bar(
    moto_num_by_decade, 
    y='first_use_date', 
    x='moto_num',  
    title="Number of motorycles by decade of registration", 
    orientation='h'
)
fig.update_layout(              
    xaxis=dict(
        title='',
        showticklabels=False,
        range=x_axis_range
    ),
    yaxis=dict(title='', tickformat=',.0f'), 
    bargap=0.1,
    width=900,
    height=500,
    margin=dict(
        l=10, 
        r=50, 
        t=50, 
        b=0, 
        pad=5
    )
)
fig.update_traces(
    texttemplate='%{x:.0f}', 
    textposition='outside', 
    hoverinfo='skip',  
    hovertemplate=None,
    textfont_size=10,
    textfont_color='#6B7280'
)

fig.show()

The data are clearly right-skewed: the older the motorcycle, the fewer observations there are. It is curious that for bin "2020-2021" we have only two years, it takes almost 1/3 of motos in 2010-2020. We can assume new motorcycles are bought actively in recent years.

### Fail rate by motorcycles' year of registration

Next we'll check does the fail rate deviate between different years of motos' registration.

In [101]:
first_use_year = first_test['first_use_date'].dt.year.rename('year')
fail_rate_by_first_use_year = fail_rate_calc(
    first_test, 
    first_use_year,
    test_num=200, 
    sort_col='year',
    to_drop=False
)
fail_rate_by_first_use_year['year'] = fail_rate_by_first_use_year['year'].astype('Int32')
fail_rate_by_first_use_year['fail_rate_ma'] = fail_rate_by_first_use_year['fail_rate'].rolling(5, min_periods=1).mean().round(4)

Only since 1976 there are more than 200 motorcycles that were registered per year, so our analysis will focus on them, small samples give us unstable estimates, huge uncertainty and unreliable comparisons.

In [102]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=fail_rate_by_first_use_year['year'],
        y=fail_rate_by_first_use_year['fail_rate'],
        mode='lines',
        name="fail rate",
        hovertemplate='Year: %{x}<br>Fail rate: %{y:.2%}<extra></extra>' 
    )
)

fig.add_trace(
    go.Scatter(
        x=fail_rate_by_first_use_year['year'],
        y=fail_rate_by_first_use_year['fail_rate_ma'],
        mode='lines',
        name="5-year MA",
        line=dict(color='rgba(99, 110, 250, 0.4)', dash='dot'),
        hovertemplate='Year: %{x}<br>MA: %{y:.2%}<extra></extra>'
    )
)
fig.update_layout(
        width=1050,
        height=500,
        title=dict(text='First test fail rate by year of registration', y=0.9),
        xaxis=dict(title=dict(text='')),
        yaxis=dict(title=dict(text=''))
)
fig.update_yaxes(tickformat=".1%")
fig.show()

The fail rate for vehicles put into use between 1993 and 2016 is fairly similar. After this period, the fail rate drops - and surprisingly, it's also lower for vehicles registered before 1993.

### Fail rate by mileage

Now we check how mileage influences the fail rate. Will we get expected results? We will round mileage to nearest 1000 and drop groups that count lower than 30.

In [103]:
fail_rate_by_mileage = fail_rate_calc(
    first_test, 
    first_test['test_mileage'].round(-3),
    sort_col='test_mileage',
    to_drop=False
)
fail_rate_by_mileage = fail_rate_by_mileage.loc[lambda x: x['test_num'] >= 30]

In [104]:
fig = px.scatter(
    fail_rate_by_mileage, 
    x='test_mileage',
    y='fail_rate', 
    trendline="lowess",
    title="Fail rate by mileage",
    color='test_num',
    hover_data={
        'test_mileage': True,
        'test_num': True,
        'fail_rate': ':.2%'
    }
)
fig.data[-1].line.color = '#9BA3AE'
fig.data[-1].opacity = 0.7
fig.update_layout(
    xaxis=dict(
        title='',
        range=[0, fail_rate_by_mileage['test_mileage'].max() * 1.05] 
    ),
    yaxis=dict(
        title='',
        tickformat=".0%",
    ),
    width=900,
    height=500
)
fig.show()

We could say the trend is non-linear, it sharply rises until 40k, then the slope flattens out and the curve plateaus around 10-11% from roughly 60k-80k. From 80k trend line starts to drop slowly, in the same time, points start to spread, at 90-100k they scatter wildly (from ~3% up to 21%) compared to the tight clustering below 40k. As we see on hover, with higher mileage, number of tests per point is getting smaller. As we round points to the nearest thousands, values equal or less than 500 will round to 0, and such point differs from the trend and has higher fail rate. It requires some further investigation. Next we check how many motorcycles travel less than 200 miles per year and visualize it.

In [105]:
lt_200_miles_per_year = first_test.loc[(first_test['test_mileage'] / first_test['age']) <= 200].shape[0] / first_test.shape[0]

fig = px.histogram(
    first_test.loc[(first_test['test_mileage'] / first_test['age']) <= 200, 'age'], 
    title='Number of motorcycles having less than 200 miles per year by age',
    nbins=20,
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(title='age'),
    yaxis=dict(title='count', tickformat=',0f'),
    title=dict(y=0.9),
    showlegend=False
)               
fig.show()

4.91% of motorcycles travel less than 200 miles per year.

### Fail rate by brand

Many motorcycle brands appear in the dataset. To compare fail rates by brand, brands with fewer than 1000 tests are excluded.

In [106]:
fail_rate_by_make = fail_rate_calc(
    first_test, 
    first_test['make'],
    test_num=None, 
    sort_col='fail_rate',
    to_drop=False
)
fail_rate_by_pop_make = (
    fail_rate_by_make.loc[lambda x: x['test_num'] >= 1000]
    .sort_values(by='fail_rate')
    .reset_index(drop=True)
) 

In [107]:
fig = px.bar(
    fail_rate_by_pop_make, 
    y='make', 
    x='fail_rate',  
    title="First test fail rate by brand", 
    orientation='h',
    hover_name="make", 
    hover_data={
        "fail_rate": ':.2%',
        "test_num": True,
        "make": False,
    },
)
fig.update_layout(
    xaxis=dict(
        title='',
        showticklabels=True,
        tickformat='.0%',
    ),
    yaxis=dict(
        title='',
        tickfont=dict(size=10)
    ),
    bargap=0.1,
    width=900,
    height=500,
    margin=dict(
        l=10, 
        r=50, 
        t=50, 
        b=0, 
        pad=5)
)
mean_fail_rate = first_test['passed'].value_counts(normalize=True)[False]
fig.add_vline(
    x=mean_fail_rate, 
    line_dash="dash", 
    line_color="#FFC107",
    annotation_text=f" Mean: {mean_fail_rate:.2%}",
    annotation_font_size=10,
    annotation_position="bottom right"
)
                       
fig.show()

Fail rate ranges from nearly 3% to 23% for brands number of tests exceed 1000, we can see there are some clusters of brands with similar fail rate.

### Popular brands fail rate by age

Next we compare top 10 brands(by number of tests) fail rate by motorcycles' age

In [108]:
top10_makers = first_test['make'].value_counts()[:10].index.to_list()
top10_makers_first_test = first_test.loc[first_test['make'].isin(top10_makers)]
year_bins = pd.cut(
    top10_makers_first_test['age'], 
    bins=[0, 5, 10, 15, 20, 30, 40, 120], 
    labels=['0-5', '5-10', '10-15', '15-20', '20-30', '30-40', '40+'],
    right=False
)
top10_makers_by_age = fail_rate_calc(
    top10_makers_first_test, 
    ['make', year_bins],
    test_num=200,
    sort_col=['make', 'age'],
    to_drop=False
)

In [109]:
fig = px.line(
    top10_makers_by_age, 
    x='age', 
    y='fail_rate', 
    color='make',
    title='Top 10 brands first test fail rate by age',
    markers=True,
    hover_data={
        "test_num": True,
        'fail_rate': ':.2%'
    },
    category_orders={'make': top10_makers},
)
default_visible_brands = {'HONDA', 'YAMAHA', 'SUZUKI'}
fig.update_layout(
    width=1050,
    height=500,
    xaxis=dict(title=''),
    yaxis=dict(title='', tickformat=".0%"),
    title=dict(y=0.9)
)
fig.update_yaxes(rangemode="tozero")
for trace in fig.data:
    if trace.name not in default_visible_brands:
        trace.visible = 'legendonly' 
fig.show()

We have increasing fail rate over years though the peak is different for all brands: "20-30" for 4 brands, "15-20" for 3 brands, "10-15" for 2 brands, and "30-40" for one brand. After the peak fail rate drops but for 4 brands it's increasing again.

### Popular brand fail rate by mileage 

Does the fail rate deviation depends on mileage for popular brands? We create custom bins to group mileage.

In [110]:
top10_or_other_brand = pd.Series(np.where(first_test['make'].isin(top10_makers), first_test['make'], 'OTHER BRANDS'))
top10_or_other_brand.name = 'top10_or_other_brand'
mileage_bins = pd.cut(
    first_test['test_mileage'], 
    [0, 10000, 30000, 60000, 90000, np.inf],
    right=False, 
    labels=["0-10000", "10000-30000", "30000-60000", "60000-90000","90000+"]
)
top10_brand_by_mileage = fail_rate_calc(
    first_test, 
    [top10_or_other_brand, mileage_bins],
    test_num=30,
    sort_col=['top10_or_other_brand', 'test_mileage'],
    to_drop=False
)

In [111]:
fig = px.line(
    top10_brand_by_mileage, 
    x='test_mileage', 
    y='fail_rate', 
    color='top10_or_other_brand',
    title='Top 10 brands first test fail rate by mileage',
    markers=True,
     hover_data={
        "test_num": True,
         "fail_rate": ':.2%'
    },
    category_orders={'top10_or_other_brand': top10_makers+["OTHER BRANDS"]}
)
fig.update_layout(
    width=1050,
    height=500,
    legend_title_text='brand',
    xaxis=dict(title='', showticklabels=True),
    yaxis=dict(
        title='',
        tickformat=".0%",
    ),
    title=dict(y=0.9)
)
for trace in fig.data:
    if trace.name not in default_visible_brands:
        trace.visible = 'legendonly' 
fig.show()

For all brands fail rate is getting higher with more miles covered, but for half of top brands it's rising when they get 90000+ miles, for Triuph and Piaggio the increase is quite sharp, for other half it drops for last bin, sharply for Kawasaki, but the usual case for our dataset is small tests number of last bins.

### Fail rate by cylinder capacity

Then we check another guess, does cylinder capacity has any influence on test result? Cylinder capacity bins are created and used to group it.

In [112]:
first_test_cc_bins = pd.cut(
    first_test["cylinder_capacity"],
    bins=[40, 125, 250, 400, 500, 750, 1000, 1500, 2000, 2500],
    labels=["40-125", "125-250", "250-400", "400-500", "500-750", "750-1000", "1000-1500", "1500-2000", "2000-2500"],
    include_lowest=True,
    right=True
)
fail_rate_by_cc_bins = fail_rate_calc(
    first_test, 
    first_test_cc_bins,
    sort_col='cylinder_capacity',
    to_drop=False
)

In [113]:
fig = px.bar(
    fail_rate_by_cc_bins, 
    x='cylinder_capacity',
    y='fail_rate', 
    title="First test fail rate by cylinder capacity", 
    hover_data={
        "fail_rate": ':.2%',
        "test_num": True,
    },
)
fig.update_layout(
    width=900,
    height=500,
    title=dict(y=0.9),
    xaxis=dict(
        title='',
    ),
    yaxis=dict(
        title='',
        tickformat='.0%'
    ),
)
fig.show()

We clearly see that fail rate is getting lower for higher cc, however 500-750 cc bin doesn't follow this tendency. First bin, 40-125,  is twice as higher as each bin, except for forementioned 500-750. Let's create a scatterplot, but this time we round cc to nearest ten.

In [114]:
cc_rounded = first_test['cylinder_capacity'].round(-1)
fail_rate_by_cc = fail_rate_calc(
    first_test, 
    cc_rounded,
    test_num=30,
    sort_col='cylinder_capacity',
    to_drop=False
)

Again, we drop groups of cylinder capacity which has less than 30 tests, then we visualize it using scatter plot with trendine. We take a log of number of tests to highlight points with low/high test number.

In [115]:
fail_rate_by_cc['log_test_num'] = np.log10(fail_rate_by_cc['test_num']).round(2)

fig = px.scatter(
    fail_rate_by_cc, 
    x='cylinder_capacity',
    y='fail_rate', 
    trendline="ols",
    title="Fail rate by cylinder capacity",
    color='log_test_num',
    hover_data={
        'cylinder_capacity': True,
        'test_num': True,
        'fail_rate': ':.2%',
        'log_test_num': False
    }
)
fig.data[-1].line.color = '#9BA3AE'
fig.data[-1].opacity = 0.7
fig.update_layout(
    xaxis=dict(
        title='',
        range=[0, fail_rate_by_cc['cylinder_capacity'].max() * 1.05] 
    ),
    yaxis=dict(
        title='',
        tickformat=".0%",
    ),
    width=900,
    height=500
)
fig.show()

A pattern can be observed where motorcycles with higher cylinder capacity have lower fail rate. Points scatter almost uniformly for the middle of the chart. Again, we clearly see cc 50, 100-120 has much higher fail rate. As well, there are two points(1020 and 1770) which have 0% fail rate, however their test number is relatively small.

In [116]:
pop_make_mean_cc = (first_test.groupby('make', as_index=False)
    .agg(
        make_count=('make', 'count'), 
        mean_cc=('cylinder_capacity', 'mean')
    )
    .loc[lambda x: x['make_count'] >= 1000]
    .reset_index(drop=True))
pop_make_mean_cc.merge(fail_rate_by_pop_make, on='make')[['make', 'mean_cc', 'fail_rate']].corr(numeric_only=True)

,mean_cc,fail_rate
mean_cc,1.000000,-0.704937
fail_rate,-0.704937,1.000000


We see here strong negative association between two measures that supports previous possible explanation of the tendency. 

### Top 10 brands first test fail rate by cylinder capacity

In [117]:
top10_brand_by_cc = (
    first_test.groupby(
        [top10_or_other_brand, first_test_cc_bins], 
        as_index=False,
        observed=False)['passed']
    .value_counts()
    .pivot_table(
        columns='passed', 
        values='count', 
        index=['top10_or_other_brand', 'cylinder_capacity'],
        fill_value=0,
        observed=False)
    .rename(columns={True: 'passed', False: 'failed'})
    .loc[lambda x: x['passed'] + x['failed'] >= 200]
    .astype('Int32') 
    .reset_index()
)
top10_brand_by_cc.columns.name = None
top10_brand_by_cc['test_num'] = top10_brand_by_cc['failed'] + top10_brand_by_cc['passed']
top10_brand_by_cc['fail_rate'] = (top10_brand_by_cc['failed'] / top10_brand_by_cc['test_num']).round(4) 

In [118]:
fig = px.line(
    top10_brand_by_cc, 
    x='cylinder_capacity', 
    y='fail_rate', 
    color='top10_or_other_brand',
    title='Top 10 brands first test fail rate by cylinder capacity',
    markers=True,
    category_orders={'top10_or_other_brand': top10_makers+["OTHER BRANDS"]},
    hover_data={"test_num": True, "fail_rate": ":.2%"}
)
fig.update_layout(
    width=1050,
    height=500,
    legend_title_text='brand',
    xaxis=dict(title=''),
    yaxis=dict(
        title='',
        tickformat=".0%",
        range=[0, 0.25]
    ),
    title=dict(y=0.9),
    margin=dict(pad=5)
)

fig.update_xaxes(
    categoryorder='array',
    categoryarray=first_test_cc_bins.cat.categories.tolist()
)
for trace in fig.data:
    if trace.name not in default_visible_brands:
        trace.visible = 'legendonly'
fig.show()

All brands if they have motos with "40-125" cc, have highest fail rate. Honda's "40-125" cc fail rate (18.34%) is dramatically higher than every other brand in that bin - even KTM and Yamaha at ~15% are notably lower. Suzuki, Kawasaki, and Piaggio all show a local peak around 400-500cc rather than a smooth monotonic decline - interrupting clean downward trend.

### Fail rate by region and cylinder capacity

We saw fail rate for both categories separately, now we find out does fail rate deviate for various cylinder capacity groups across UK regions.

In [119]:
fail_rate_by_region_cc = (
    first_test.groupby(
        [first_test_regions, first_test_cc_bins], 
        as_index=False,
        observed=False)['passed']
    .agg(
        fail_rate=lambda x: (x == False).sum() / len(x),
        test_num='count')
    .merge(
        fail_rate_by_region[['region','fail_rate']],
        left_on='region', 
        right_on='region',
        suffixes=('', '_by_region'))
    .merge(
        fail_rate_by_cc_bins[['cylinder_capacity','fail_rate']],
        how='left',
        left_on='cylinder_capacity', 
        right_on='cylinder_capacity',
        suffixes=('', '_by_cc')
    ) 
    .rename(columns={'postcode_area': 'region'})
    .sort_values(
        by=['fail_rate_by_region', 'cylinder_capacity'], 
        ascending=[False, True])
    .reset_index(drop=True)
)


In [120]:
fig = px.bar(
    fail_rate_by_region_cc, 
    x='region', 
    y='fail_rate', 
    title='First test fail rate by region & cylinder capacity',
    color='cylinder_capacity', 
    barmode="group",
    hover_data={
        'fail_rate': ':.2%',
        'test_num': True,
        'fail_rate_by_region': ':.2%',
        'fail_rate_by_cc': ':.2%'
    }
)
fig.update_layout(
    width=1050,
    height=500,
    legend_title_text='cylinder_capacity',
    xaxis=dict(
        title='',
        tickfont=dict(
            size=11 
        )
    ),
    yaxis=dict(
        title='',
        tickfont=dict(
            size=11
        ),
        tickformat=".1%"
    ),
    title=dict(
        y=0.9
    ),
)
default_visible_cc = {'40-125', '125-250', '250-400', '400-500', '500-700'}
for trace in fig.data:
    if trace.name not in default_visible_cc:
        trace.visible = 'legendonly'
fig.show()

In hover mode we see not only fail rate by region & cylinder capacity together, but separate measures for region and cc. Bar chart is sorted by fail rate of region descending, so we clearly see the difference in fail rate for particular cylinder capacity between regions. South West's 40-125cc bar hits 21.26%, notably higher than every other region's small-capacity bar (most cluster 12-17%). South West's other bars (500-750cc especially, ~11%) are also elevated relative to other regions. The only bar where South West is not top 1 is 2000-2500cc however it still takes 2nd place. Greater London is a clear outlier in the opposite direction — much lower fail rates across the board. The 40-125cc bar (10.15%) is roughly half of South West's, and its other categories are also consistently among the lowest of all regions except 125-400 & 2000-2500. This is an interesting contrast with possible explanations: London's motorcycle fleet may skew toward newer bikes, different usage patterns (shorter average trips vs. long-distance rural riding elsewhere), stricter/different maintenance culture, or a different mix of vehicle types (e.g., more delivery bikes that might be newer/well-maintained fleet vehicles vs. older privately-owned bikes elsewhere) and overall comfort city enviroment. Accordingly 40-125 has highest number of motorcycles for this region. Interestingly number of 2000-2500 cc motorcycles is smallest in Greater London, maybe this kind of motos isn't quite appropriate for everyday city use. Its fail rate is lowest on north and east middle part of UK. 

### Fail rate by decade of registration and cylinder capacity

In [121]:
fail_rate_by_decade_cc = (
    first_test.groupby(
        [decade_bins, first_test_cc_bins], 
        as_index=False,
        observed=False)['passed']
    .value_counts()
    .pivot_table(
        columns='passed', 
        values='count', 
        index=['first_use_date', 'cylinder_capacity'],
        fill_value=0,
        observed=False)
    .rename(columns={True: 'passed', False: 'failed'})
    .loc[lambda x: x['passed'] + x['failed'] >= 200]
    .astype('Int32') 
    .reset_index()
)

fail_rate_by_decade_cc.columns.name = None
fail_rate_by_decade_cc['test_num'] = fail_rate_by_decade_cc['failed'] + fail_rate_by_decade_cc['passed']
fail_rate_by_decade_cc['fail_rate'] = (fail_rate_by_decade_cc['failed'] / fail_rate_by_decade_cc['test_num']).round(4) 
fail_rate_by_decade_cc.sort_values(by=['first_use_date', 'cylinder_capacity'], inplace=True)

In [122]:
fig = px.bar(
    fail_rate_by_decade_cc, 
    x='first_use_date', 
    y='fail_rate', 
    title='First test fail rate by decade of registration & cylinder capacity',
    color='cylinder_capacity', 
    barmode="group",
    category_orders={
        'first_use_date': ["1960-1970","1970-1980","1980-1990","1990-2000", "2000-2010", "2010-2020", "2020-2021"],
        'cylinder_capacity': ['40-125', '125-250', '250-400', '400-500', '500-750', '750-1000', '1000-1500', '1500-2000', '2000-2500']},
    hover_data={
        'fail_rate': ':.2%',
        'test_num': True
    }
)
fig.update_layout(
    width=1050,
    height=500,
    legend_title_text='cylinder_capacity',
    xaxis=dict(
        title='',
        tickfont=dict(
            size=12 
        )
    ),
    yaxis=dict(
        title='',
        tickfont=dict(
            size=12 
        ),
        tickformat=".1%"
    ),
    title=dict(
        y=0.9
    ),
)
default_visible_cc = {'40-125', '125-250', '250-400', '400-500', '500-700'}
for trace in fig.data:
    if trace.name not in default_visible_cc:
        trace.visible = 'legendonly'
fig.show()

If we look from older to newer motorcycles, for most bins, fail rate starts to grow until certain point. It's not the case for motorcycles with high cc: fail rate for "1500-2000" and "2000-2500" drops consistently, but these bins have enough number of tests starting from 1990-2000 & 2000-2010 accordingly. From 2000 fail rate for bin "40-125" differs from other bins of the same decade significantly. The spread between smallest and largest capacity has grown substantially:
Tight ~4-8% band in 1980-1990 vs. a much wider ~4-15% range by 2010-2020 — reinforcing that the small-capacity penalty has intensified over time, not stayed constant.

### Fail rate by decade of registration & mileage

In [123]:
fail_rate_by_decade_mileage = (
    first_test.groupby(
        [decade_bins, mileage_bins], 
        as_index=False,
        observed=False)['passed']
    .value_counts()
    .pivot_table(
        columns='passed', 
        values='count', 
        index=['first_use_date', 'test_mileage'],
        fill_value=0,
        observed=False)
    .rename(columns={True: 'passed', False: 'failed'})
    .loc[lambda x: x['passed'] + x['failed'] >= 30]
    .astype('Int32') 
    .reset_index()
)

fail_rate_by_decade_mileage.columns.name = None
fail_rate_by_decade_mileage['test_num'] = fail_rate_by_decade_mileage['failed'] + fail_rate_by_decade_mileage['passed']
fail_rate_by_decade_mileage['fail_rate'] = (fail_rate_by_decade_mileage['failed'] / fail_rate_by_decade_mileage['test_num']).round(4) 
fail_rate_by_decade_mileage.sort_values(by=['first_use_date', 'test_mileage'], inplace=True)

In [124]:
fig = px.bar(
    fail_rate_by_decade_mileage, 
    x='first_use_date', 
    y='fail_rate', 
    title='First test fail rate by decade of registration & test mileage',
    color='test_mileage', 
    barmode="group",
    category_orders={
        'first_use_date': ["1950-1960", "1960-1970", "1970-1980","1980-1990","1990-2000", "2000-2010", "2010-2020", "2020-2021"],
        'test_mileage': ["0-10000", "10000-30000", "30000-60000", "60000-90000", "90000+"]},
    hover_data={
        'fail_rate': ':.2%',
        'test_num': True
    }
)
fig.update_layout(
    width=1050,
    height=500,
    legend_title_text='test_mileage',
    xaxis=dict(
        title='',
        tickfont=dict(
            size=12 
        )
    ),
    yaxis=dict(
        title='',
        tickfont=dict(
            size=12 
        ),
        tickformat=".1%"
    ),
    title=dict(
        y=0.9
    ),
)
fig.show()

This time we visualize only bins where test number exceeds 30. Within every decade, fail rate rises with mileage - confirms the pattern holds across all eras. Bars step upward left to right in almost every decade, consistent across 70+ years of registration data. Low-mileage bikes (up to 10000) stay relatively flat across decades - 0.9% for 1950-1960 but for next decades it is from 3.84% to 6.44%, while high-mileage bikes show a much steeper rise — from 2.65-3.8% in 1960-1970 to ~9.3-14.68% by 2010-2021. Consequently spread within each decade is widening: for 1980-1990 the spread is 1.95%, for 2010-2020 the spread is 8.86%. The mileage-related gap keeps growing decade over decade. Worth noticing that at least one mileage group is absent for first 3 decades(1950-1960, 1960-1970, 1970-1980). We can conclude that most vehicles registered before 1980 aren't used widely and can be collectible ones. Interestingly, fail rate for motorcycles with mileage 60000-90000 registered in 1970-1980 is very low(1.89%) however the sample size is 53. Again we can assume these bikes are collectibles. 

## Restructuring 'Item Group' table

To perform a deeper analysis, the `Item Group(ig)` table is restructured. The reason is `['item_id']` may refer to both single part and section. Next it will be merged with `Item Detail(itde)` table for further analysis.

In [125]:
ig.head()

,test_item_id,parent_id,test_item_set_section_id,item_name
0,0,0,0,Vehicle
1,0,0,0,Vehicle
6,1,0,1,Motorcycle lighting and signalling
7,1,0,1,Motorcycle lighting and signalling
8,2,1,1,Position lamps


In [126]:
#original

# moto_items = (
#     ig[['test_item_id', 'test_item_set_section_id', 'item_name']]
#     .merge(
#          ig[['test_item_id', 'item_name']], 
#          how='left',
#          suffixes=('', '_section'),
#          left_on='test_item_set_section_id', 
#          right_on='test_item_id'
#     )
#     .drop(['test_item_id_section'],axis=1)
#     .rename(columns={'item_name_section': 'section_name'})
# )
# moto_items['section_name'] = (
#     moto_items['section_name'].str.replace('motorcycle', '', case=False,  regex=True)
#     .str.strip()
#     .str.lower()
#     .str.capitalize()
# )
# moto_items_rfrs = (
#     itde.merge(
#         moto_items, 
#         how='left',
#         on=['test_item_id', 'test_item_set_section_id']
        
#     )
#     .rename(columns={'test_item_id': 'item_id', 'test_item_set_section_id': 'section_id'})
# )
# moto_items_rfrs.head()

In [127]:
moto_items = (
    ig
    .merge(
         ig, 
         how='left',
         suffixes=('', '_section'),
         left_on='test_item_set_section_id', 
         right_on='test_item_id'
    )
    [['test_item_id', 'test_item_set_section_id', 'item_name', 'item_name_section']]
    .rename(columns={
        'item_name_section': 'section_name',
        }
    )
)
moto_items['section_name'] = (
    moto_items['section_name'].str.replace('motorcycle', '', case=False,  regex=True)
    .str.strip()
    .str.lower()
    .str.capitalize()
)
moto_items.head()

,test_item_id,test_item_set_section_id,item_name,section_name
0,0,0,Vehicle,Vehicle
1,0,0,Vehicle,Vehicle
2,0,0,Vehicle,Vehicle
3,0,0,Vehicle,Vehicle
4,1,1,Motorcycle lighting and signalling,Lighting and signalling


In [128]:
itde.head()

,rfr_id,test_item_id,minor_item,rfr_deficiency_category,rfr_desc,test_item_set_section_id
0,4,3,True,Pre-EU Directive,an obligatory lamp missing,1
2,5,3,True,Pre-EU Directive,so damaged or deteriorated that its function i...,1
4,6,3,True,Pre-EU Directive,so damaged or deteriorated that its function i...,1
6,7,3,True,Pre-EU Directive,an insecure lamp,1
8,8,3,True,Pre-EU Directive,is obscured or does not face the front or rear...,1


In [129]:
moto_items_rfrs = (
    itde.merge(
        moto_items, 
        how='left',
        on=['test_item_id', 'test_item_set_section_id']
        
    )
    .rename(columns={'test_item_id': 'item_id', 'test_item_set_section_id': 'section_id'})
    .drop_duplicates(keep='first')
)
moto_items_rfrs.head()

,rfr_id,item_id,minor_item,rfr_deficiency_category,rfr_desc,section_id,item_name,section_name
0,4,3,True,Pre-EU Directive,an obligatory lamp missing,1,Motorcycle front,Lighting and signalling
4,5,3,True,Pre-EU Directive,so damaged or deteriorated that its function i...,1,Motorcycle front,Lighting and signalling
8,6,3,True,Pre-EU Directive,so damaged or deteriorated that its function i...,1,Motorcycle front,Lighting and signalling
12,7,3,True,Pre-EU Directive,an insecure lamp,1,Motorcycle front,Lighting and signalling
16,8,3,True,Pre-EU Directive,is obscured or does not face the front or rear...,1,Motorcycle front,Lighting and signalling


# 4. Detailed analysis

## Merging tables

Next we merge all tables to have detailed analysis of tests results with failed parts and type of malfunction.

In [130]:
ti.head()

,test_id,rfr_id,rfr_type_code,completed_date
0,939637859,40098,A,2024-01-02 10:03:29
1,401677457,40450,A,2024-01-02 10:47:37
2,401677457,40546,A,2024-01-02 10:47:37
3,401677457,40663,A,2024-01-02 10:47:37
4,401677457,40547,A,2024-01-02 10:47:37


In [131]:
test_items_rfrs = ti.merge(moto_items_rfrs, on='rfr_id', how='left')
test_items_rfrs.head()

,test_id,rfr_id,rfr_type_code,completed_date,item_id,minor_item,rfr_deficiency_category,rfr_desc,section_id,item_name,section_name
0,939637859,40098,A,2024-01-02 10:03:29,25016,True,Dangerous,Brake lining or pad worn below 1.0mm,25003,Brake pads,Brakes
1,401677457,40450,A,2024-01-02 10:47:37,25085,True,Major,A shock absorber not functioning or leaking se...,25081,Shock absorbers,Suspension
2,401677457,40546,A,2024-01-02 10:47:37,25108,True,Major,Exhaust system leaking or insecure,25105,Exhaust system,Structure and attachments
3,401677457,40663,A,2024-01-02 10:47:37,25108,False,Major,Exhaust noise levels in excess of those permitted,25105,Exhaust system,Structure and attachments
4,401677457,40547,A,2024-01-02 10:47:37,25108,True,Major,Exhaust system leaking or insecure,25105,Exhaust system,Structure and attachments


In [132]:
test_items_rfrs.isna().sum()

test_id                      0
rfr_id                       0
rfr_type_code                0
completed_date               0
item_id                      0
minor_item                   0
rfr_deficiency_category      0
rfr_desc                     0
section_id                   0
item_name                  160
section_name               160
dtype: int64

We have some missing values for `['section_name']` and `['item_name']` columns after the merge, but `['rfr_desc']` aren't empty for those rows. They exist in `Item Group` table, but belong not to motorcycle classes (1,2) and duplicate existing section and items. We'll leave it as it is.

In [133]:
rfr_desc_missing_rfr_id = test_items_rfrs.loc[test_items_rfrs['rfr_desc']=="Tyre tread depth not in accordance with the requirements", 'rfr_id'].value_counts()
rfr_desc_missing_rfr_id

rfr_id
40428    59608
40427    15045
31194        9
31193        6
31195        3
Name: count, dtype: Int64

In [134]:
moto_items_rfrs.loc[moto_items_rfrs['rfr_id'].isin(rfr_desc_missing_rfr_id.index)]

,rfr_id,item_id,minor_item,rfr_deficiency_category,rfr_desc,section_id,item_name,section_name
8208,31193,20233,False,Dangerous,Tyre tread depth not in accordance with the re...,20449,<NA>,<NA>
8209,31194,20233,False,Dangerous,Tyre tread depth not in accordance with the re...,20449,<NA>,<NA>
8210,31195,20233,False,Dangerous,Tyre tread depth not in accordance with the re...,20449,<NA>,<NA>
10712,40427,25104,True,Dangerous,Tyre tread depth not in accordance with the re...,25102,Condition,Tyres
10716,40428,25104,True,Dangerous,Tyre tread depth not in accordance with the re...,25102,Condition,Tyres


In [135]:
display(Markdown(f"Unique motorcycle parts: {(test_items_rfrs['section_name'] + '-' + test_items_rfrs['item_name']).nunique()}"))
display(Markdown(f"Unique motorcycle sections: {test_items_rfrs['section_id'].nunique()}"))
display(Markdown(f"Unique reasons for rejection(rfr): {test_items_rfrs['rfr_id'].nunique()}"))

Unique motorcycle parts: 87

Unique motorcycle sections: 19

Unique reasons for rejection(rfr): 685

In [136]:
display(Markdown(f"Each unique reason for rejection(rfr) refers to:"))
test_items_rfrs.groupby('rfr_deficiency_category', observed=True, dropna=False)['rfr_id'].nunique()

Each unique reason for rejection(rfr) refers to:

rfr_deficiency_category
Dangerous           176
Major               436
Minor                58
Pre-EU Directive     15
Name: rfr_id, dtype: int64

In [137]:
display(Markdown(f"Is motorcycle defect considered as 'minor' (qualifies for free partial retest): "))
test_items_rfrs.groupby(['rfr_id'], as_index=False)['minor_item'].value_counts()['minor_item'].value_counts()

Is motorcycle defect considered as 'minor' (qualifies for free partial retest): 

minor_item
True     557
False    128
Name: count, dtype: int64

In [138]:
failed_items = (
    test_items_rfrs
    .loc[(test_items_rfrs['rfr_deficiency_category'].isin(['Dangerous', 'Major'])) 
        & (test_items_rfrs['rfr_type_code']=='F')]
     
)
rfr_num_cause_f = failed_items['rfr_id'].nunique() 
display(Markdown(f"Number of rfrs which explicitly cause the test failure: {rfr_num_cause_f}"))

Number of rfrs which explicitly cause the test failure: 571

In [139]:
failed_items_num_by_section = failed_items.groupby('section_name')['rfr_id'].nunique().sort_values(ascending=False)
display(Markdown(f"Number of rfrs which explicitly cause the test failure by section:"))
failed_items_num_by_section

Number of rfrs which explicitly cause the test failure by section:

section_name
Brakes                           157
Suspension                        86
Structure and attachments         82
Lamps and reflectors              73
Steering                          59
Wheels                            29
Tyres                             19
Identification of the vehicle     11
Audible warning (horn)             2
Name: rfr_id, dtype: int64

In [140]:
first_test_items_rfrs = (
    first_test.merge(
        test_items_rfrs, 
        on=['test_id', 'completed_date'], 
        how='inner')
    .drop(['test_id'], axis=1)
)
first_test_items_rfrs.head()

,vehicle_id,test_result,test_mileage,postcode_area,make,model,cylinder_capacity,first_use_date,completed_date,age,passed,rfr_id,rfr_type_code,item_id,minor_item,rfr_deficiency_category,rfr_desc,section_id,item_name,section_name
0,73,P,29024,SA,HONDA,UNCLASSIFIED,249,1991-03-22,2024-07-13 10:54:40,33.31,True,10013,A,10000,True,Pre-EU Directive,Chain guard missing,10000,Non-component advisories,Non-component advisories
1,73,P,29024,SA,HONDA,UNCLASSIFIED,249,1991-03-22,2024-07-13 10:54:40,33.31,True,40391,A,25101,True,Major,A wheel hub excessively worn or damaged,25098,Hub,Wheels
2,782,P,17755,B,HONDA,CBR1100X-2,1137,2003-06-04,2024-07-25 13:10:18,21.14,True,40450,A,25085,True,Major,A shock absorber not functioning or leaking se...,25081,Shock absorbers,Suspension
3,13284,PRS,33733,LE,YAMAHA,FZS600,599,1999-11-24,2024-07-15 12:02:36,24.64,True,40669,PRS,25072,False,Dangerous,A stop lamp(s) does not illuminate by the oper...,25061,Stop lamp,Lamps and reflectors
4,20755,F,4659,AL,KAWASAKI,ZX,599,2014-07-01,2024-08-14 08:14:46,10.12,False,40323,F,25072,True,Major,"Stop lamp missing, inoperative or in the case ...",25061,Stop lamp,Lamps and reflectors


After merge, we can add some details to summary analysis of first test

In [141]:
first_test_moto_num = first_test['vehicle_id'].nunique()
test_result_values = first_test['test_result'].value_counts()
motos_passed_flawlessly = first_test_moto_num-first_test_items_rfrs["vehicle_id"].nunique()
motos_passed_minor_defects = test_result_values["P"] - motos_passed_flawlessly
motos_passed_prs = test_result_values["PRS"]
motos_failed = test_result_values["F"]
motos_aborted = test_result_values["ABR"]
motos_abandoned = test_result_values["ABA"]
data = {
    "count": [
        motos_passed_flawlessly,
        motos_passed_minor_defects,
        motos_passed_prs,
        motos_failed,
        motos_aborted,
        motos_abandoned
    ]
}

index = [
    "passed test flawlessly",
    "passed test with minor defects",
    "passed after rectification",
    "failed test",
    "aborted test",
    "abandoned test"
]

first_test_stats = pd.DataFrame(data, index=index)
first_test_stats["percentage(%)"] = (first_test_stats["count"] / first_test_stats['count'].sum() * 100).round(2)

first_test_stats.loc[first_test_stats.shape[0]+1] = first_test_stats.sum(axis=0).round()
first_test_stats = first_test_stats.rename(index={7: 'Total'})
first_test_stats['count'] = first_test_stats['count'].astype('Int32')
display(Markdown(f"Results of the first test:"))
first_test_stats

Results of the first test:

,count,percentage(%)
passed test flawlessly,540246,65.73
passed test with minor defects,172266,20.96
passed after rectification,46630,5.67
failed test,62496,7.60
aborted test,208,0.03
abandoned test,38,0.00
Total,821884,100.00


Now we depict detailed picture of first test result, using `['rfr_deficiency_category']`, i.e how many motorcycle parts were into various rfr categories. It means we don't count motorcycles that passed the test flawlessly.

In [142]:
## Motorcycles that passed the test but has "F" for "Major" и "Dangerous" ['rfr_deficiency_category']

# first_test_items_rfrs.loc[
#     (first_test_items_rfrs['rfr_deficiency_category']=='Major') 
#     & (first_test_items_rfrs['rfr_type_code']=='F') 
#     & (first_test_items_rfrs['passed']==True), 
#     'vehicle_id'
# ].unique()

In [143]:
first_test_num_rfr_type = first_test_items_rfrs[['rfr_deficiency_category', 'rfr_type_code', 'passed', 'vehicle_id']]

first_test_num_rfr_type_pivot = (
    first_test_num_rfr_type.pivot_table(
        index=['rfr_deficiency_category', 'rfr_type_code'], 
        columns='passed', 
        values='vehicle_id', 
        aggfunc='count',
        fill_value=0, 
        observed=True,
        
    )
)
first_test_num_rfr_type_pivot['failed_pct(%)'] = (first_test_num_rfr_type_pivot[False] / first_test_num_rfr_type_pivot[False].sum() * 100).round(2)
first_test_num_rfr_type_pivot['passed_pct(%)'] = (first_test_num_rfr_type_pivot[True] / first_test_num_rfr_type_pivot[True].sum() * 100).round(2)
first_test_num_rfr_type_pivot['passed_mean_moto_parts'] = (
    first_test_num_rfr_type_pivot[True]
    / first_test_items_rfrs.loc[first_test_items_rfrs['passed']==True, 'vehicle_id'].nunique()
    ).round(2)

first_test_num_rfr_type_pivot['failed_mean_moto_parts'] = (
    first_test_num_rfr_type_pivot[False] 
     / first_test_items_rfrs.loc[first_test_items_rfrs['passed']==False, 'vehicle_id'].nunique()
    ).round(2)
                                            
grand = first_test_num_rfr_type_pivot.sum(axis=0).to_frame().T
grand.index = pd.MultiIndex.from_tuples([('All', 'All')])  
first_test_num_rfr_type_pivot =  pd.concat([first_test_num_rfr_type_pivot, grand]).rename(columns={True: 'passed', False: 'failed'})
first_test_num_rfr_type_pivot = first_test_num_rfr_type_pivot[['passed', 'passed_pct(%)', 'passed_mean_moto_parts', 'failed', 'failed_pct(%)', 'failed_mean_moto_parts']]

first_test_num_rfr_type_pivot[['passed', 'failed']] = first_test_num_rfr_type_pivot[['passed', 'failed']].astype('Int32')
first_test_num_rfr_type_pivot.columns.name = None
first_test_num_rfr_type_pivot.iloc[-1,-2] = first_test_num_rfr_type_pivot.iloc[-1,-2].round(0)
first_test_num_rfr_type_pivot

passed  passed_pct(%)  passed_mean_moto_parts  failed  \
Dangerous        A     81225          23.65                    0.37   14698   
                 F         0            0.0                     0.0   30692   
                 PRS    8491           2.47                    0.04     173   
Major            A    192842          56.15                    0.88   41889   
                 F         0            0.0                     0.0   97840   
                 PRS   47266          13.76                    0.22    1436   
Minor            A        75           0.02                     0.0       7   
                 M      9681           2.82                    0.04    4257   
Pre-EU Directive A      3880           1.13                    0.02    1370   
All              All  343460          100.0                    1.57  192362   

                      failed_pct(%)  failed_mean_moto_parts  
Dangerous        A             7.64                    0.24  
                 F            15.96                    0.49  
                 PRS           0.09                     0.0  
Major            A            21.78                    0.67  
                 F            50.86                    1.57  
                 PRS           0.75                    0.02  
Minor            A              0.0                     0.0  
                 M             2.21                    0.07  
Pre-EU Directive A             0.71                    0.02  
All              All          100.0                    3.08

As expected, motos that passed the test have zero parts with mark "F" in "Dangerous" and "Major" category in `['failed']` column (that come from test rules), meanwhile they have more marks "A" and "PRS", meaning their condition was good enough not to fail the test / quick repair was possible. On average each failed moto has 2.06 (1.57+0.49) parts thas cause the failure. We can add that parts with mark "Dangerous" occur less often than "Major". "Minor" & "Pre-EU Directive" category takes only around 4% (3.97% and 3.92%) for passed and failed parts. Overall motorcycles that failed the test, have as twice as more parts with issues (3.08) than motorcycles that passed the test (1.57).

### Defect items by section, deficiency category & type code

From our merged table we see what flaws motorcycles have for first test, but we don't have summary table for all items, how often they occur and to what deficiency category they belong.

In [144]:
rfr_type_code_by_section_item_cat = first_test_items_rfrs.groupby(
    ['section_name', 'item_name', 'rfr_deficiency_category', 'rfr_type_code'], 
    as_index=False, 
    observed=False
).size()
rfr_type_code_by_section_item_cat_pivot = pd.pivot_table(
    rfr_type_code_by_section_item_cat, 
    index=['section_name', 'item_name'], 
    columns=['rfr_deficiency_category','rfr_type_code'], 
    values='size', 
    aggfunc='sum', 
    observed=False, 
    margins=True,
    margins_name='Total'
)

rfr_type_code_by_section_item_cat_pivot = (
    rfr_type_code_by_section_item_cat_pivot
        .loc[lambda x: x['Total'] > 0]
        .sort_values(by='Total', ascending=False)
        .iloc[1:, :]
        .pipe(lambda x: x.loc[:, (x != 0).any(axis=0)])
        .sort_values(by=['section_name', 'Total', 'item_name'], ascending=[True, False, True])
        .reset_index()
)

html_parts = []

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for section in rfr_type_code_by_section_item_cat_pivot['section_name'].unique():
        subset = (rfr_type_code_by_section_item_cat_pivot.loc[rfr_type_code_by_section_item_cat_pivot['section_name'] == section]
                    .set_index(['section_name', 'item_name']))
        subset = subset.droplevel('section_name')
        
        subset.columns.names = [None, 'item_name']  
        subset.index.name = None  
        
        html_parts.append(f"<h3>{section}</h3>")
        html_parts.append(subset.to_html(classes='mytable'))

display(HTML("""
    <style>
        .mytable { width: 100% !important; table-layout: fixed !important; border-collapse: collapse !important; margin-bottom: 20px !important; }
        .mytable thead tr th:not(:first-child) { text-align: center !important; padding: 4px 8px !important; border-left: 1px solid rgba(128,128,128,0.2) !important; border-right: 1px solid rgba(128,128,128,0.2) !important; border-top: 1px solid rgba(128,128,128,0.2) !important; border-bottom: 1px solid rgba(128,128,128,0.2) !important; color: inherit !important; background-color: transparent !important; }
        .mytable thead tr th:first-child { border: 1px solid rgba(128,128,128,0.2) !important; padding: 4px 8px !important; }
        .mytable th, .mytable td { padding: 4px 8px !important; overflow: hidden !important; color: inherit !important; background-color: transparent !important; }
        .mytable th:first-child, .mytable td:first-child { text-align: left !important; }
        .mytable td { text-align: center !important; }
    </style>
""" + "\n".join(html_parts)))

#### Rfr deficiency categories
According to [official MOT website](https://www.gov.uk/guidance/mot-inspection-manual-for-motorcycles/introduction), various defect categories may be available for the same item depending on the nature or severity of the defect – minor, major or dangerous. Just looking at columns filling we can conclude the less count "Minor" category has per item, the more severe item defect could be.

##### Rfr deficiency category "Pre-EU directive"
From our table we see that this category has its own section "Non-component advisories" and has no rfr type code except for "A". If we check rfr description for this section or item it returns "np.nan". We can conclude from the title that it's some common advices about vehicle conditions.

##### Rfr deficiency category "Minor"
Many items have 0 for both rfr type codes, meaning that severity of damage is higher or nature is different. Only "Audible warning (horn)" section has non-zero count for "A" in the category.

##### Rfr_type_code "A" & "M" 
if the item has "A" for "Dangerous" rfr deficiency category it won't have "M" for "Minor" but the item has "A" for "Major" it can have "A" for "Minor". Possibly it can be explained that nature of the damage is closer between "Minor" & "Major" categories.

##### Rfr_type_code "PRS"
Only 4 items in "Major" and 2 items in "Major" category have more "PRS" count than "F", meaning they are easy to fix despite referring to more "severe" category.

##### Rfr_type_code "F" for "Dangerous" category
For all sections we have one item that has much higher count than others.

##### Rfr_type_code "F" for "Major" category
As well as "F" for "Dangerous" category, "Major" category has one leader, except for "Lamp and reflectors" sections, high count items are distributed more uniformly.

#### Sections
"Brakes", "Steering", "Structure and attachments", "Suspension", "Tyres" and "Wheels" are categories that almost have no count for "Minor" rfr deficiency category, so the damage is more severe for them. "Lamps and reflectors" is the only section where values for rfr type code "F" has uniformly distibuted count. For other sections 1-3 items have much higher count than the rest. "Lamps and reflectors" also has only one item for "Dangerous" category - "stop lamp", as we noticed before, high number of "PRS" for this section means that most of the items are easy and fast to fix.

Now we group results of the first test by various columns to create treemap to highlight existing hierarchies and analyze distribution in branches and nodes.

In [145]:
rfr_num_by_type_section_item = (
    first_test_items_rfrs.groupby(
        [   
            'passed',
            'rfr_type_code', 
            'section_name', 
            'item_name', 
            'rfr_desc'
        ], 
        observed=True, 
        as_index=False)
    .size()
    .sort_values(by='size', ascending=False)
)
rfr_num_by_type_section_item['rfr_type_code'] = rfr_num_by_type_section_item['rfr_type_code'].astype(str)

rfr_num_by_type_section_item['tr_pct'] = (
    rfr_num_by_type_section_item.groupby('passed')['size'].transform('sum')
    / rfr_num_by_type_section_item['size'].sum()
).round(4)

rfr_num_by_type_section_item['rfr_type_pct'] = (
    rfr_num_by_type_section_item.groupby(['passed','rfr_type_code'])['size'].transform('sum')
    / rfr_num_by_type_section_item.groupby('passed')['size'].transform('sum')
).round(4)

rfr_num_by_type_section_item['section_pct_by_rfr_type'] = (
    rfr_num_by_type_section_item.groupby(['passed', 'rfr_type_code', 'section_name'])['size'].transform('sum')
    / rfr_num_by_type_section_item.groupby(['passed','rfr_type_code'])['size'].transform('sum')
).round(4)

rfr_num_by_type_section_item['item_pct_by_section'] = (
    rfr_num_by_type_section_item.groupby(['passed', 'rfr_type_code', 'section_name', 'item_name'])['size'].transform('sum')
    / rfr_num_by_type_section_item.groupby(['passed','rfr_type_code', 'section_name'])['size'].transform('sum')
).round(4)

rfr_num_by_type_section_item['rfr_pct_by_section'] = (
    rfr_num_by_type_section_item.groupby(['passed', 'rfr_type_code', 'section_name', 'rfr_desc'])['size'].transform('sum')
    / rfr_num_by_type_section_item.groupby(['passed','rfr_type_code', 'section_name'])['size'].transform('sum')
).round(4)

rfr_num_by_type_section_item['rfr_pct_by_rfr_type'] = (
    rfr_num_by_type_section_item.groupby(['passed', 'rfr_desc']) ['size'].transform('sum')
    / rfr_num_by_type_section_item.groupby(['passed', 'rfr_type_code'])['size'].transform('sum')
).round(4)

In [146]:
fig = px.treemap(
    rfr_num_by_type_section_item,
    path=[
        'passed',
        'rfr_type_code', 
        'section_name', 
        'item_name', 
        'rfr_desc'
    ], 
    values='size',
    title='Number of rfrs by test result & section',
    color='rfr_type_code',
    color_discrete_map={
        "A": "#a6bddb",  
        "F":  "#E8A5A5",   
        "PRS": "#1c9099",  
        "M": "#99d8c9"  
    },
    hover_data={
        'size': True,
        'tr_pct': True, 
        'rfr_type_pct': True,
        'section_pct_by_rfr_type': True,
        'item_pct_by_section': True,
        'rfr_pct_by_section': True,
        'rfr_pct_by_rfr_type': True
    }
)
fig.update_layout(
    width=900, 
    height=900, 
    title=dict(y=0.95)
)
fig.update_traces(
    hovertemplate=
        "<b>%{label}</b><br>" +
        "Size: %{customdata[0]}<br>" +
        "Test result percentage: %{customdata[1]:.2%}<br>" +
        "Rfr type percentage by test result: %{customdata[2]:.2%}<br>" +
        "Section percentage by rfr type: %{customdata[3]:.2%}<br>" +
        "Item percentage by section: %{customdata[4]:.2%}<br>" +
        "Rfr percentage by section: %{customdata[5]:.2%}<br>" +
        "Rfr percentage by rfr type: %{customdata[6]:.2%}<br>" +
        "<extra></extra>"
)
marker_colors = list(fig.data[0]['marker']['colors'])
marker_colors[:2] = ["#e6e6e6", "#e6e6e6"]
fig.data[0]['marker']['colors'] = tuple(marker_colors)

fig.show() 

Almost 36% of defect items 'failed' first test, more than 2/3 of which explicitly cause the failure. As we see items that belong to 'passed' category, predominantly consist of "A" deficiency category (80.73%). Items that were rectified within an hour("PRS" category) take 16.42%. Around 1/3 of failed items in "F" category (33.39%) take "Lamps and reflectors" section, "Tyre tread depth not in accordance with requirements" is most frequent rfr amongst all sections. Now we create bar chart with number of parts per section for failed motorcycles. 

In [220]:
#вернуть, если где-то взята первая переменная

# first_test_failed = first_test_items_rfrs.loc[first_test_items_rfrs['test_result']=='F']
# first_test_failed_items = first_test_failed.loc[first_test_failed['rfr_type_code']=='F'].reset_index(drop=True)
first_test_failed_items = first_test_items_rfrs.loc[lambda x: (x['test_result']=='F') & (x['rfr_type_code']=='F')].reset_index(drop=True)

rfr_num_by_section = first_test_failed_items['section_name'].value_counts().reset_index()
rfr_num_by_section['pct'] = (rfr_num_by_section['count'] / rfr_num_by_section['count'].sum()).round(4)
rfr_num_by_section['cumsum_pct'] = rfr_num_by_section['pct'].cumsum().round(4)
rfr_num_by_section.iloc[6, 0] = 'Identification<br> of the vehicle'
rfr_num_by_section.iloc[7, 0] = 'Audible warning<br> (horn)	'

In [221]:
fig = px.bar(
    rfr_num_by_section, 
    x="section_name", 
    y="count",
    title="Number of failed parts by section",
    hover_data={
        'count': ':.0f',
        'pct': ':.2%'
    }
)

fig.add_trace(
    go.Scatter(
        x=rfr_num_by_section["section_name"], 
        y=rfr_num_by_section["cumsum_pct"],
        mode="lines+markers",
        name="Cumulative sum",
        yaxis="y2",
        hovertemplate="Category: %{x}<br>Cumulative sum: %{y:.2%}<extra></extra>"
    )
)
fig.update_layout(
    xaxis=dict(
        title="", 
        showticklabels=True,
        tickfont=dict(size=11)
        ),
    yaxis=dict(
        title="",
        tickformat=',.0f',
        tickfont=dict(size=11)),
    yaxis2=dict(
        title="",
        showgrid=False,
        overlaying="y",
        side="right",
        range=[0, 1],
        tickformat=".0%",
        tickfont=dict(size=11)
    ),
    bargap=0.1,
    width=1200,
    height=500,
    margin=dict(l=10, r=50, t=50, b=0, pad=5),
    legend=dict(
        x=0.9,
        y=0.92,
        orientation="h",
        xanchor="center",
        yanchor="top", 
    ),
)
fig.update_xaxes(
    tickangle=0,
    automargin=True
)
fig.show()

As we confirmed before,  "Lamps and reflectors" takes the lead with almost 43000 failed parts, it's 1.5 times more than "Brakes" section (27165). These sections account more than 50% of all rfrs. It follows Pareto distribution: a small number of categories drive the overwhelming majority of failures.

### Top 20 rfrs of rectified items

In MOT testing data guide value "PRS" from `['rfr_type_code']` column means that item was in a failing state at the point of test, repaired within one hour of the test and before the result was entered. Let's find out top 20 rfrs for this code type.

In [222]:
top20_prs_rfrs = (
    first_test_items_rfrs.loc[first_test_items_rfrs['rfr_type_code']=='PRS']
    .groupby(['section_name', 'item_name'], as_index=False)['rfr_desc'].value_counts()
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)
top20_prs_rfrs.index = top20_prs_rfrs.index + 1
with pd.option_context('display.max_colwidth', None):
    display(top20_prs_rfrs.head(20))

,section_name,item_name,rfr_desc,count
1,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,9989
2,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,8223
3,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",3289
4,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",2510
5,Identification of the vehicle,Registration plate,Number plate does not conform to the specified requirements,2397
6,Audible warning (horn),Motorcycle audible warning (Horn),Audible warning not working,2352
7,Lamps and reflectors,Registration plate lamp,A rear registration plate lamp or light source missing or inoperative in the case of a single lamp or all lamps,1977
8,Lamps and reflectors,Position lamp,A lamp missing or inoperative,1868
9,Lamps and reflectors,Individual direction indicators,"A direction indicator lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",1795
10,Lamps and reflectors,Stop lamp,A stop lamp(s) does not illuminate by the operation of both brake controls or remains on when the brakes are released,1790


As code type description suggests, we clearly see here easy-to-fix defects, 8 of top 10 belongs to "Lamps and reflectors" section.

### Relationship between number of failed parts and chance to pass the test

Next we find out does number of failed items influence the chance to pass the test eventually.

In [223]:
rfr_per_moto = first_test_failed_items.groupby('vehicle_id')['rfr_id'].agg(rfr_num='count')
failed_parts_num_by_moto_count = (
    rfr_per_moto['rfr_num'].value_counts()
    .sort_index()
    .reset_index()
    .loc[lambda x: x['count']>= 10]
)

In [224]:
failed_parts_num_by_moto_count['pass_ratio'] = 0.0
for rfr_num in range(1, failed_parts_num_by_moto_count['rfr_num'].max() + 1):
    try:
        moto_ids = rfr_per_moto[rfr_per_moto['rfr_num'] == rfr_num].index
        moto = (
            tr.loc[tr['vehicle_id'].isin(moto_ids)]
            .sort_values(by=['vehicle_id', 'completed_date'])
            .groupby('vehicle_id', as_index=False)
            .last()
        )
        pct = (moto['passed'].value_counts(normalize=True)[True]).round(2)
        failed_parts_num_by_moto_count.at[rfr_num-1, 'pass_ratio'] = pct
    except Exception as e:
        continue

In [225]:
fig = px.line(
    failed_parts_num_by_moto_count, 
    x='rfr_num', 
    y='pass_ratio', 
    title='Percentage of motos eventually passed the test with number of failed parts',
    hover_data={'count': True}
)
fig.update_layout(
    width=1050,
    height=500,
    xaxis=dict(
        title=''
    ),
    yaxis=dict(
        title='',
        tickformat=".0%",
        range=[0, 1]
    ),
    title=dict(y=0.9),
    margin=dict(pad=20)
)
fig.show()

We clearly see that chance to pass the test is dropping with increasing number of failed parts for motorcycle.

### Title

desc

In [226]:
motos_id_that_have_two_tests = (
    tr.sort_values(by=['vehicle_id', 'completed_date'])
    .groupby('vehicle_id')
    .nth(1)
    ['vehicle_id']
)
motos_first_two_tests = tr.loc[tr['vehicle_id'].isin(motos_id_that_have_two_tests)]
motos_first_two_tests['test_number'] = motos_first_two_tests.groupby('vehicle_id').cumcount() + 1
def busday_diff(date_list):
    for idx in range(len(date_list) - 1):
        start = date_list[idx].to_numpy().astype('datetime64[D]')
        end = date_list[idx+1].to_numpy().astype('datetime64[D]')
        diff = np.busday_count(start, end, weekmask=[1,1,1,1,1,1,0])
    return diff
days_num_between_tests = (
    motos_first_two_tests.groupby('vehicle_id')['completed_date'].agg(list)
        .apply(busday_diff)
        .reset_index(name='days_num')
)
moto_ids_with_dangerous_defects = (
    first_test_items_rfrs.loc[
        (first_test_items_rfrs['rfr_type_code'] == 'F') 
        & (first_test_items_rfrs['rfr_deficiency_category'] == 'Dangerous'),
        'vehicle_id'
        ]
)

paid_retest = days_num_between_tests.loc[days_num_between_tests['days_num'] > 10].shape[0]
free_retest = days_num_between_tests.loc[days_num_between_tests['days_num'] <= 10].shape[0]
free_retest_dng_defect = (
    days_num_between_tests.loc[
        (days_num_between_tests['days_num'] <= 10) 
        & (days_num_between_tests['vehicle_id'].isin(moto_ids_with_dangerous_defects))
        ].shape[0]
)
display(
    Markdown(
        f"{free_retest} of {days_num_between_tests.shape[0]} motos got no fee or "
        f"partial fee retest, {free_retest_dng_defect} have dangerous defects and can't be taken away"
    )
)

54476 of 56772 motos got no fee or partial fee retest, 21800 have dangerous defects and can't be taken away

### Why some motos didn't take second attempt to pass the test?

Although the number of motorcycles that did not take a second test is relatively small (5.9k), their rfrs are analyzed by ranking them and comparing them with motorcycles that passed the first test in order to identify severe defects that may discourage owners from retaking the test.

In [227]:
moto_failed_first_test_vals = first_test.loc[first_test['passed']==False, 'vehicle_id'].values
second_test_moto_vals = (
    tr.sort_values(by=['vehicle_id', 'completed_date'])
    .groupby('vehicle_id')
    .nth(1)['vehicle_id']
    .values
)
moto_abandonded_second_test = tr.loc[
    (tr['vehicle_id'].isin(moto_failed_first_test_vals)) & (~tr['vehicle_id'].isin(second_test_moto_vals))
]

moto_abandonded_second_test_items_rfrs = (
    moto_abandonded_second_test.merge(
        test_items_rfrs, 
        on=['test_id', 'completed_date'], 
        how='left'
    )
    .drop(['test_id', 'test_type'], axis=1)
    .loc[lambda x: x['rfr_type_code']=='F']
)
moto_abandonded_second_test_reasons = (
    moto_abandonded_second_test_items_rfrs.groupby(
        [
        'rfr_deficiency_category', 
         'section_name', 
        'item_name', 
        'rfr_desc'
        ], 
        as_index=False, 
        observed=True
    )
    .size()
)
moto_abandonded_second_test_reasons['size'] = moto_abandonded_second_test_reasons['size'].astype('Int64')

In [228]:
moto_having_second_test_first_test_results = (
    tr.loc[tr['vehicle_id'].isin(second_test_moto_vals)]
    .sort_values(by=['vehicle_id', 'completed_date'])
    .groupby('vehicle_id')
    .nth(0)
)
moto_having_second_test_first_test_items_rfrs = (
    moto_having_second_test_first_test_results.merge(
        test_items_rfrs, 
        on=['test_id', 'completed_date'], 
        how='left'
    )
    .drop(['test_id', 'test_type'], axis=1)
    .loc[lambda x: x['rfr_type_code']=='F']
)
moto_having_second_test_first_test_reasons = (
    moto_having_second_test_first_test_items_rfrs.groupby([
        'rfr_deficiency_category', 
        'section_name', 
        'item_name', 
        'rfr_desc'
        ], 
        as_index=False, 
        observed=True
    ).size()
)

In [229]:
def rank_cols(df):
    df = df.copy()
    num_cols = df.select_dtypes(include='number')
    num_cols_ranked = num_cols.rank(method='dense', ascending=False).astype('int64')
    df[num_cols.columns] = num_cols_ranked
    return df

In [230]:
continued_vs_abandoned = (
    moto_abandonded_second_test_reasons.merge(
        moto_having_second_test_first_test_reasons, 
        how='outer', 
        on=['rfr_deficiency_category', 'section_name', 'item_name', 'rfr_desc'], 
        suffixes=('_abandoned', '_continued')
    )
)
continued_vs_abandoned[['size_abandoned', 'size_continued']] = continued_vs_abandoned[['size_abandoned', 'size_continued']].fillna(0)
continued_vs_abandoned = continued_vs_abandoned.sort_values(by='size_abandoned', ascending=False)
continued_vs_abandoned = continued_vs_abandoned.reset_index(drop=True)
continued_vs_abandoned.index = continued_vs_abandoned.index + 1
continued_vs_abandoned_ranked = rank_cols(continued_vs_abandoned)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Rfrs of motorcycles took / abandoned second test:"))
    display(continued_vs_abandoned_ranked.head(10))

Rfrs of motorcycles took / abandoned second test:

,rfr_deficiency_category,section_name,item_name,rfr_desc,size_abandoned,size_continued
1,Major,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,1,2
2,Dangerous,Tyres,Condition,Tyre tread depth not in accordance with the requirements,2,1
3,Major,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",3,8
4,Major,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",4,4
5,Major,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,5,6
6,Dangerous,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,6,3
7,Major,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,7,7
8,Major,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",8,5
9,Major,Brakes,Roller brake test,Significant brake effort recorded with no brake applied indicating a binding brake,9,11
10,Major,Lamps and reflectors,Individual direction indicators,"A direction indicator lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",10,12


In [231]:
#original

# continued_vs_abandoned = (
#     moto_abandonded_second_test_reasons.merge(
#         moto_having_second_test_first_test_reasons, 
#         how='outer', 
#         on=['rfr_deficiency_category', 'section_name', 'item_name', 'rfr_desc'], 
#         suffixes=('_abandoned', '_continued')
#     )
# )
# continued_vs_abandoned[['size_abandoned', 'size_continued']] = continued_vs_abandoned[['size_abandoned', 'size_continued']].fillna(0)
# continued_vs_abandoned = continued_vs_abandoned.sort_values(by='size_abandoned', ascending=False)
# continued_vs_abandoned = filter_below_threshold(continued_vs_abandoned)
# continued_vs_abandoned.index = continued_vs_abandoned.index + 1
# continued_vs_abandoned_ranked = rank_cols(continued_vs_abandoned)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{continued_vs_abandoned_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Rfrs of motorcycles took / abandoned second test:"))
#     display(continued_vs_abandoned_ranked.head(10))

In [232]:
def filter_below_threshold(df, thres):
    num_cols = df.select_dtypes(include='number').columns
    pct_cumsum = df[num_cols].cumsum() / df[num_cols].sum(axis=0)
    mask = ~(pct_cumsum >= thres).all(axis=1).shift(fill_value=False)
    df = df[mask].reset_index(drop=True)
    return df

As expected, most defects belong to the "Major" category, and 5 of the top 10 rfrs fall within the "Lamps and reflectors" section. The only notable rank difference for a top 10 appears for the defect "A transmission belt, chain, sprocket, or pulley excessively loose or worn." To measure similarity between defect distributions, we will use correlation function.

In [233]:
def corr_df(df):
    res = {}
    for thres in [1, 0.75, 0.5]:
        df_filt = filter_below_threshold(df, thres)
        corr = df_filt.corr(numeric_only=True).mean()
        res[f"{thres:.0%}"] = corr
    
    corr_table = pd.DataFrame(res)
    corr_table = corr_table.sort_values(by=corr_table.columns[0]) 
    return corr_table

In [234]:
corr_df(continued_vs_abandoned)

,100%,75%,50%
size_abandoned,0.960848,0.904141,0.832849
size_continued,0.960848,0.904141,0.832849


In [235]:
def js_distance(df):
    num_cols = df.select_dtypes(include='number').columns
    df = df[num_cols]
    n = len(num_cols)

    result = pd.DataFrame(
        np.zeros((n, n)),
        index=num_cols,
        columns=num_cols
    )

    for i in range(n):
        for j in range(i, n):

            p = df.iloc[:, i]
            q = df.iloc[:, j]

            dist = distance.jensenshannon(p, q)

            result.iat[i, j] = dist
            result.iat[j, i] = dist

    return result

In [236]:
js_dist_matrix = js_distance(continued_vs_abandoned)
js_dist_matrix

,size_abandoned,size_continued
size_abandoned,0.000000,0.138727
size_continued,0.138727,0.000000


As we see, correlation is very strong, so it's hard to highlight any particular reason for not retaking the test.

### Hard-to-repair motorcycle parts

Next we check the parts that failed during second test i.e. weren't fixed after first test.

In [237]:
second_test_items_rfrs = (
    tr.groupby('vehicle_id')
    .nth(1)
    .merge(
        test_items_rfrs, 
        on=['test_id', 'completed_date'], 
        how='inner'
    )
    .drop(['test_id', 'test_type'], axis=1)
)
display(Markdown(f"Top 10 items & its rfr that are hard to fix: "))
hard_fixable_items = (
    second_test_items_rfrs.loc[second_test_items_rfrs['rfr_type_code']=='F']
    .merge(
        first_test_items_rfrs,
        on=['vehicle_id', 'rfr_type_code', 'item_id', 'section_id'],
        how='inner',
        suffixes=('_first', '_second')
    )
    .rename(columns={
            'section_name_first': 'section_name', 
            'item_name_first': 'item_name', 
            'rfr_desc_first': 'rfr_desc'}
    )
    .groupby([
        'section_name', 
        'item_name', 
        'rfr_desc'
        ],
        as_index=False
    )
    .size()
    .sort_values(by='size', ascending=False)
    .reset_index(drop=True)
)
hard_fixable_items.index = hard_fixable_items.index + 1
with pd.option_context('display.max_colwidth', None):
    display(hard_fixable_items.head(10))

Top 10 items & its rfr that are hard to fix: 

,section_name,item_name,rfr_desc,size
1,Brakes,Roller brake test,Excessive fluctuation in brake effort through each wheel revolution,12
2,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,7
3,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,7
4,Brakes,Roller brake test,Significant brake effort recorded with no brake applied indicating a binding brake,7
5,Tyres,Condition,Tyre tread depth not in accordance with the requirements,6
6,Lamps and reflectors,Headlamp aim,Projected beam image is obviously incorrect,6
7,Lamps and reflectors,Registration plate lamp,A rear registration plate lamp or light source missing or inoperative in the case of a single lamp or all lamps,5
8,Identification of the vehicle,Registration plate,Number plate does not conform to the specified requirements,5
9,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,5
10,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,5


### Proportion of "Major" vs "Dangerous" failed parts

Although "Dangerous" defects occur less frequently than "Major" defects, it is worth examining whether the proportion of these categories changes for motorcycles of different ages.

In [238]:
first_test_dang_proportion_by_year = (
    first_test_failed_items[['first_use_date', 'rfr_deficiency_category']]
    .groupby(first_test_failed_items['first_use_date'].dt.year.rename('year'), as_index=False)['rfr_deficiency_category']
    .agg(
        total_parts='count',
        dang_pct=lambda x: (x == 'Dangerous').sum() / len(x)
    )
    .loc[lambda x: x['total_parts'] > 30]
    .reset_index(drop=True)
)
first_test_dang_proportion_by_year['dang_pct_ma'] = first_test_dang_proportion_by_year['dang_pct'].rolling(5, min_periods=1).mean().round(4)

In [239]:
fig = make_subplots()
fig.add_trace(
    go.Scatter(
        x=first_test_dang_proportion_by_year['year'],
        y=first_test_dang_proportion_by_year['dang_pct'],
        name='Dangerous parts %', 
        mode='lines',
        hovertemplate='<b>%{x}</b><br>Dangerous %: %{y:.2%}<extra></extra>' 
    ),
)

fig.add_trace(
    go.Scatter(
        x=first_test_dang_proportion_by_year['year'],
        y=first_test_dang_proportion_by_year['dang_pct_ma'],
        name='5-year Moving Average',  
        mode='lines',
        line=dict(
            dash='dash',
            color='rgba(99, 110, 250, 0.4)'
        ),
        hovertemplate='<b>%{x}</b><br>Dangerous % 5-Year MA: %{y:.2%}<extra></extra>'  
    ),
)

fig.update_layout(
    width=1050,
    height=500,
    title=dict(
        text="Proportion of 'Dangerous' parts by vehicles' year of registration",
        y=0.9
    ),
    xaxis=dict(
        title=''
    )
)

fig.update_yaxes(
    title_text='',
    showticklabels=True,
    tickformat=".0%",
)

fig.show()

We see here interesting tendency: share of "Dangerous" category parts is higher for newer vehicles. Then we compare fail rate & "Dangerous" category share by popular brands.

In [240]:
first_test_dang_proportion_by_make = (
    first_test_failed_items.groupby('make', as_index=False)['rfr_deficiency_category']
    .agg(dang_pct=lambda x: (x == 'Dangerous').sum() / len(x),
         parts_num='count'
        )
    .loc[lambda x: x['parts_num'] >=200]
    .reset_index(drop=True)
    .merge(fail_rate_by_make[['make', 'fail_rate']], on='make', how='left')
)

In [241]:
fig = px.scatter(
    first_test_dang_proportion_by_make,
    x="fail_rate",
    y="dang_pct",
    title="Fail rate vs proportion of dangerous parts by brand",
    text="make",
    trendline="ols", 
    hover_name="make",  
    hover_data={
        "make": False,
        "parts_num": ":.0f",
        "fail_rate": ":.1%",
        "dang_pct": ":.1%"
    },
)

fig.update_layout(
    xaxis=dict(
        title="Fail rate",
    ),
    yaxis=dict(
        title="Proportion of dangerous parts",
        range=[0, 0.35],
    ),
    title=dict(y=0.9),
    width=900,
    height=600,
)

fig.update_traces(
    textfont=dict(size=9),
    textposition="bottom center",
)

fig.update_xaxes(
    tickformatstops=[
        dict(dtickrange=[None, 0.01], value=".1%"), 
        dict(dtickrange=[0.01, None], value=".0%")  
    ]
)

fig.update_yaxes(
    tickformatstops=[
        dict(dtickrange=[None, 0.01], value=".1%"),
        dict(dtickrange=[0.01, None], value=".0%")
    ]
)

fig.show()

Displaying both parameters on scatterplot helps us see some clusters more clearly, surprisingly, motorcycles with low fail rate (<=5%) have one of the highest parts share labeled "Dangerous". Motorcycles with highest fail rate also have quite similar dangerous parts' proportion. Next we calculate correlation of two variables.

In [242]:
df = first_test_dang_proportion_by_make[['dang_pct', 'fail_rate']]
df.corr(method='spearman')

,dang_pct,fail_rate
dang_pct,1.000000,-0.320648
fail_rate,-0.320648,1.000000


There is a negative medium association between fail rate and proportion of dangerous parts for different brands, now we find out does mileage influences percentage of "Dangerous" parts.

In [243]:
first_test_dang_proportion_by_mileage = (
    first_test_failed_items.groupby(
        mileage_bins, 
        observed=True, 
        as_index=False
    )['rfr_deficiency_category']
    .agg(
        dangerous=lambda x: (x == 'Dangerous').sum(),
        major=lambda x: (x == 'Major').sum(),
        dang_pct=lambda x: (x == 'Dangerous').sum() / len(x),
        parts_num='count'
    )
    .loc[lambda x: x['parts_num'] > 0]
    .reset_index(drop=True)
)

In [244]:
fig = px.bar(
    first_test_dang_proportion_by_mileage, 
    x='test_mileage',
    y='dang_pct', 
    title='Proportion of "Dangerous" parts by mileage',
    hover_data={"dang_pct": ":.2%"}
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(
        title='',
        showticklabels=True,
        ), 
    yaxis=dict(
        title='',
        range=[0,0.4],
        tickformat='.0%'
    ),
    bargap=0.1,
)
fig.show()

Visually there is almost no difference between bins, chi-square test is conducted to check it.

In [245]:
dang_prop_cont_table = first_test_dang_proportion_by_mileage.iloc[:,:3].set_index('test_mileage')
chi2, p_value, dof, expected = chi2_contingency(dang_prop_cont_table)

print("\nChi-Square Test results:")
print(f"Chi2 Test Statistic: {chi2}")
print(f"P-value: {p_value}")
print(f"Degrees of Freedom (dof): {dof}")

alpha = 0.05
if p_value < alpha:
    print(f"Because p-value ({p_value:.4f}) < 0.05, we reject null hypothesis.")
    print("The difference in mileage bins and its proportion of dangerous parts is statistically significant.")
else:
    print(f"Because p-value ({p_value:.4f}) >= 0.05, we fail to reject null hypothesis.")
    print("We haven't got enough evidence to say the difference between share of dangerous parts for various mileage bins is significant.")


Chi-Square Test results:
Chi2 Test Statistic: 2.3345654051214932
P-value: 0.6744831868491614
Degrees of Freedom (dof): 4
Because p-value (0.6745) >= 0.05, we fail to reject null hypothesis.
We haven't got enough evidence to say the difference between share of dangerous parts for various mileage bins is significant.


In [246]:
first_test_failed_items_cc_bins = pd.cut(
    first_test_failed_items["cylinder_capacity"],
    bins=[40, 125, 250, 400, 500, 750, 1000, 1500, 2000, 2500],
    labels=["40–125", "125–250", "250–400", "400–500", "500–750", "750–1000", "1000–1500", "1500–2000", "2000–2500"],
    include_lowest=True,
    right=True
)

first_test_dang_proportion_by_cc = (
    first_test_failed_items.groupby(
        first_test_failed_items_cc_bins, 
        observed=True, 
        as_index=False
    )['rfr_deficiency_category']
    .agg(
        dangerous=lambda x: (x == 'Dangerous').sum(),
        major=lambda x: (x == 'Major').sum(),
        dang_pct=lambda x: (x == 'Dangerous').sum() / len(x),
        parts_num='count'
    )
    .loc[lambda x: x['parts_num'] > 30]
    .reset_index(drop=True)
)

In [247]:
fig = px.bar(
    first_test_dang_proportion_by_cc, 
    x='cylinder_capacity',
    y='dang_pct', 
    title='Proportion of "Dangerous" parts by cylinder capacity',
    hover_data={"dang_pct": ":.2%"}
)
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(
        title='',
        showticklabels=True,
        ), 
    yaxis=dict(
        title='',
        range=[0,0.4],
        tickformat='.0%'
    ),
    bargap=0.1,
)
fig.show()

As we see here, motorcycles with cylinder capacity from 1000 have higher share of dangerous parts, as well it's the case for bin "40-125" but in a lower degree. Let's conduct a chi square test to check is the difference statistically significant.

In [248]:
dang_prop_cont_table = first_test_dang_proportion_by_cc.iloc[:,:3].set_index('cylinder_capacity')
chi2, p_value, dof, expected = chi2_contingency(dang_prop_cont_table)

print("\nChi-Square Test results:")
print(f"Chi2 Test Statistic: {chi2}")
print(f"P-value: {p_value}")
print(f"Degrees of Freedom (dof): {dof}")

alpha = 0.05
if p_value < alpha:
    print(f"Because p-value ({p_value:.4f}) < 0.05, we reject null hypothesis.")
    print("The difference in cylinder capacity bins and its proportion of dangerous parts is statistically significant.")
else:
    print(f"Because p-value ({p_value:.4f}) >= 0.05, we fail to reject null hypothesis.")
    print("We haven't got enough evidence to say the difference between share of dangerous parts for various cylinder capacity bins is significant.")


Chi-Square Test results:
Chi2 Test Statistic: 231.61846929809965
P-value: 1.3460610145632805e-45
Degrees of Freedom (dof): 8
Because p-value (0.0000) < 0.05, we reject null hypothesis.
The difference in cylinder capacity bins and its proportion of dangerous parts is statistically significant.


### Rfr for vehicles registered in various decades

Rfrs distributions & its ranking are compared for motorcycles registered in different decades (starting from 1980 to ensure sufficient sample sizes). We will use rank correlation to measure how defect patterns change over time. We will use Jensen-Shannon distance as well because we have to be sure that their distributions is taken into account.

In [249]:
first_use_year_bins = (
    pd.cut(
        first_test_failed_items.loc[first_test_items_rfrs['first_use_date'].dt.year >= 1980, 'first_use_date'].dt.year,
        [1980, 1990, 2000, 2010, 2020, 2022],
        right=False,
        labels=["1980-1990", "1990-2000", "2000-2010", "2010-2020", "2020-2021"])
)

rfr_comp_by_decade = (
    first_test_failed_items.groupby([
            first_use_year_bins, 
            'section_name', 
            'item_name',
            'rfr_desc'
        ], 
        as_index=False,
        observed=False
    )
    .size()
    .loc[lambda x: x['size'] > 0]
    .sort_values(by=['first_use_date', 'size'], ascending=[True, False])
    .reset_index(drop=True)
)

In [250]:
#original

# rfr_comp_by_decade_pivot = (
#     rfr_comp_by_decade.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'], 
#         columns='first_use_date',
#         values='size'
#         )
#     .sort_values(by=['2020-2021'], ascending=False)
#     .reset_index()
# )
# rfr_comp_by_decade_pivot = rfr_comp_by_decade_pivot.fillna(0.0)
# rfr_comp_by_decade_pivot.columns.name = None  
# rfr_comp_by_decade_pivot = filter_below_threshold(rfr_comp_by_decade_pivot)
# rfr_comp_by_decade_pivot.index = rfr_comp_by_decade_pivot.index + 1
# rfr_comp_by_decade_pivot_ranked = rank_cols(rfr_comp_by_decade_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{rfr_comp_by_decade_pivot_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs for decade of registration"))
#     display(rfr_comp_by_decade_pivot_ranked.head(10))

In [251]:
rfr_comp_by_decade_pivot = (
    rfr_comp_by_decade.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'], 
        columns='first_use_date',
        values='size'
        )
    .sort_values(by=['2020-2021'], ascending=False)
    .reset_index()
)
rfr_comp_by_decade_pivot = rfr_comp_by_decade_pivot.fillna(0.0)
rfr_comp_by_decade_pivot.columns.name = None  
rfr_comp_by_decade_pivot = rfr_comp_by_decade_pivot.reset_index(drop=True)
rfr_comp_by_decade_pivot.index = rfr_comp_by_decade_pivot.index + 1
rfr_comp_by_decade_pivot_ranked = rank_cols(rfr_comp_by_decade_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs by decade of registration"))
    display(rfr_comp_by_decade_pivot_ranked.head(10))

Ranking of rfrs by decade of registration

,section_name,item_name,rfr_desc,1980-1990,1990-2000,2000-2010,2010-2020,2020-2021
1,Tyres,Condition,Tyre tread depth not in accordance with the requirements,12,3,2,1,1
2,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,29,13,5,3,2
3,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,2,1,1,2,3
4,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,10,12,9,6,4
5,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",20,14,11,5,5
6,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",1,6,3,4,6
7,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,6,7,6,8,7
8,Lamps and reflectors,Registration plate lamp,A rear registration plate lamp or light source missing or inoperative in the case of a single lamp or all lamps,16,15,10,9,8
9,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",3,2,4,7,9
10,Steering,Headbearings,"Steering head bearings excessively stiff, notchy, or with excessive wear or play",22,24,21,16,10


In [252]:
# for thres in [0.9, 0.7, 0.5]:
#     rfr_comp_by_decade_pivot_filt = filter_below_threshold(rfr_comp_by_decade_pivot, thres)
#     corr = rfr_comp_by_decade_pivot_filt.corr(numeric_only=True).mean().sort_values()
#     print(f"Pearson correlation of {thres:.0%} rfrs: \n{corr.round(4)}")
#     print('')

In [253]:
# def corr_df(df):
#     res = {}
#     for thres in [1, 0.75, 0.5]:
#         df_filt = filter_below_threshold(df, thres)
#         corr = df_filt.corr(numeric_only=True).mean()
#         res[f"{thres:.0%}"] = corr
    
#     corr_table = pd.DataFrame(res)
#     corr_table = corr_table.sort_values(by=corr_table.columns[0]) 
#     return corr_table

In [254]:
corr_df(rfr_comp_by_decade_pivot)

,100%,75%,50%
1980-1990,0.867970,0.750246,0.551169
2020-2021,0.906478,0.820284,0.674504
2010-2020,0.932863,0.869599,0.764889
1990-2000,0.933611,0.869530,0.770877
2000-2010,0.952144,0.907306,0.832577


In [255]:
js_dist_matrix = js_distance(rfr_comp_by_decade_pivot)
fig = px.imshow(
        js_dist_matrix,
        width=500,
        height=450
)
fig.update_layout(
    title=dict(
        text="JS Distance of rfrs by decade of vehicles' registration",
        y=0.92,
        font=dict(size=15)
    )
)

fig.update_xaxes(tickfont=dict(size=11))
fig.update_yaxes(tickfont=dict(size=11))
fig.update_layout(
    coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=11))))
fig.show()

Due to correlation heatmap we see the longer vehicle is used the bigger difference it has comparably to newer one, i.e it has other rfrs, however overall the strength of correlation association even between 1980-1990 and 2020-2021 is medium. We can see it looking at ranks of first top 10 rfrs sorted by decade of registration descending, ranks are mostly drops over the years. But as an example, rfr "A shock absorber not functioning or leaking severely" is relevant over all years(has 1-3 place). JS distance moves aprroximately equal in opposite direction, from 1 to 0, colors of heatmaps shows us that the distributions don't differ much.

### Comparison of top failed parts across leading motorcycle brands

Then we try to figure out is there any difference between rfrs for top 10 brands, i.e do various brands have specific weak spots.

In [183]:
#??????? почему не удалил не помню

# # def filter_below_threshold(df):
# #     num_cols = df.select_dtypes(include='number').columns
# #     mask = ~((df[num_cols] / df[num_cols].sum(axis=0)) < 0.001).all(axis=1)
# #     df = df[mask].reset_index(drop=True)
# #     return df

# def filter_below_threshold(df):
#     num_cols = df.select_dtypes(include='number').columns
#     cum_pct = df[num_cols].cumsum() / df[num_cols].sum(axis=0)
#     mask = ~(cum_pct >= 0.5).all(axis=1).shift(fill_value=False)
#     df = df[mask].reset_index(drop=True)
#     return df


In [256]:
top_make_rfr_comp = (
    first_test_failed_items.loc[first_test_failed_items['make'].isin(top10_makers)]
    .groupby(
        ['make', 'section_name', 
        'item_name', 'rfr_desc'], 
        as_index=False,
        observed=False,
    )
    .size()
    .sort_values(by=['make', 'size'], ascending=[True, False])
)

In [257]:
#original

# top_make_rfr_comp_pivot = (
#     top_make_rfr_comp.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'
#         ],
#         columns='make',
#         values='size'
#     )
#     [top10_makers]
#     .sort_values(by='HONDA', ascending=False)
#     .fillna(0)
#     .reset_index()
# )
# top_make_rfr_comp_pivot.columns.name = None

# top_make_rfr_comp_pivot = filter_below_threshold(top_make_rfr_comp_pivot)
# top_make_rfr_comp_pivot.index = top_make_rfr_comp_pivot.index + 1
# top_make_rfr_comp_pivot_ranked = rank_vals(top_make_rfr_comp_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{top_make_rfr_comp_pivot.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs for top 10 brands"))
#     display(top_make_rfr_comp_pivot_ranked.head(10))

In [258]:
top_make_rfr_comp_pivot = (
    top_make_rfr_comp.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'
        ],
        columns='make',
        values='size'
    )
    [top10_makers]
    .sort_values(by='HONDA', ascending=False)
    .fillna(0)
    .reset_index()
)
top_make_rfr_comp_pivot.columns.name = None
top_make_rfr_comp_pivot = top_make_rfr_comp_pivot.reset_index(drop=True)
top_make_rfr_comp_pivot.index = top_make_rfr_comp_pivot.index + 1
top_make_rfr_comp_pivot_ranked = rank_cols(top_make_rfr_comp_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs for top 10 brands"))
    display(top_make_rfr_comp_pivot_ranked.head(10))

Ranking of rfrs for top 10 brands

,section_name,item_name,rfr_desc,HONDA,YAMAHA,SUZUKI,TRIUMPH,KAWASAKI,BMW,HARLEY-DAVIDSON,PIAGGIO,KTM,DUCATI
1,Tyres,Condition,Tyre tread depth not in accordance with the requirements,1,1,1,1,1,1,2,1,9,2
2,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,2,2,2,3,2,3,15,3,5,5
3,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,3,3,4,2,3,2,6,5,12,14
4,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",4,9,3,11,5,11,5,6,7,6
5,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",5,4,7,15,6,12,41,57,5,11
6,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,6,6,12,7,11,5,7,4,10,12
7,Lamps and reflectors,Stop lamp,A stop lamp(s) does not illuminate by the operation of both brake controls or remains on when the brakes are released,7,14,14,14,13,13,9,8,17,11
8,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",8,5,11,10,12,6,10,2,4,8
9,Brakes,Roller brake test,Significant brake effort recorded with no brake applied indicating a binding brake,9,10,6,9,8,7,19,11,16,18
10,Brakes,Roller brake test,Brake efficiency below minimum requirement,10,22,19,21,26,18,18,18,20,3


In [259]:
corr_df(top_make_rfr_comp_pivot)

,100%,75%,50%
HARLEY-DAVIDSON,0.783949,0.587382,0.408807
KTM,0.790402,0.556985,0.299659
PIAGGIO,0.808228,0.627176,0.556499
DUCATI,0.820904,0.675901,0.487506
BMW,0.854549,0.732728,0.653975
HONDA,0.868472,0.730515,0.657055
YAMAHA,0.897930,0.786840,0.726497
TRIUMPH,0.898516,0.795838,0.713544
KAWASAKI,0.898674,0.793357,0.705776
SUZUKI,0.903979,0.799584,0.707410


In [188]:
# brands_spearman_matrix = top_make_rfr_comp_pivot.corr(method='spearman', numeric_only=True)
# brands_spearman_matrix.mean().sort_values()

In [260]:
brands_js_dist_matrix = js_distance(top_make_rfr_comp_pivot)
font_size = 12
fig = px.imshow(
        brands_js_dist_matrix,
        width=700,
        height=700
)
fig.update_layout(
    title=dict(
        text="JS Distance of rfrs by top 10 brands",
        y=0.93,
        font=dict(size=15)
    )
)

fig.update_xaxes(tickfont=dict(size=font_size))
fig.update_yaxes(tickfont=dict(size=font_size))
fig.update_layout(
    coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
fig.show()

In [190]:
# brands_js_dist_matrix = js_distance(top_make_rfr_comp_pivot)
# brands_spearman_matrix = top_make_rfr_comp_pivot.corr(method='spearman', numeric_only=True)
# fig = make_subplots(
#     rows=1, cols=2,
#     subplot_titles=("JS Distance", "Spearman Correlation")
# )

# fig_js = px.imshow(
#         brands_js_dist_matrix,
#         width=700,
#         height=700, 
# )
# fig_spearman = px.imshow(
#         brands_spearman_matrix,
#         width=700,
#         height=700
# )

# fig.add_trace(fig_js.data[0], row=1, col=1)
# fig.add_trace(fig_spearman.data[0], row=1, col=2)

# fig.data[0].update(coloraxis="coloraxis1")
# fig.data[1].update(coloraxis="coloraxis2")

# fig.update_layout(
#     coloraxis1=dict(colorscale='Plasma', colorbar=dict(x=0.45, tickfont=dict(size=11))),
#     coloraxis2=dict(colorscale='Plasma_r', colorbar=dict(x=1.0, tickfont=dict(size=11))),
#     height=500, width=1100,
#     title_text="JS Distance vs Spearman Correlation of rfrs across top 10 brands"
# )

# fig.update_xaxes(tickfont=dict(size=11))
# fig.update_yaxes(tickfont=dict(size=11), row=1, col=1)
# fig.update_yaxes(showticklabels=False, row=1, col=2, tickfont=dict(size=11))
# 
# fig.show()

The mean of correlation shows that Piaggio rfr ranking differs the most from other brands but mean js distance shows there are 3 brands (Piaggio, Ducati, Harley-Davidson) have comparably equal rfr distribution. The lowest correlation occurs between Ducati and Piaggio (0.57), while the highest occurs between Suzuki and Kawasaki (0.95). The lowest js distance is between Ducati and Piaggio as well (0.36). A cluster of brands with very similar defect profiles can also be observed: Honda, Yamaha, Suzuki, Triumph, Kawasaki, and BMW, with lowest correlation association between Honda and BMW (0.8301). Looking at difference between rank correlation & (1-js distance) Piaggio is the only brand with negative value, i.e. rfr distribution is more similar than for other brands.

### Rfrs comparison for different mileage

Now we check whether motos with different mileage have the same rfrs.

In [191]:
first_test_failed_items_mileage_bins = pd.cut(
    first_test_failed_items['test_mileage'], 
    [0, 10000, 30000, 60000, 90000, np.inf],
    right=False, 
    labels=["0-10000", "10000-30000", "30000-60000", "60000-90000","90000+"]
)

In [192]:
rfr_comp_by_mileage = (
    first_test_failed_items.groupby(
        [first_test_failed_items_mileage_bins, 'section_name', 'item_name', 'rfr_desc'], 
        observed=False, 
        as_index=False
    )
    .size()
    .loc[lambda x: x['size'] > 0]
    .sort_values(by=['test_mileage', 'size'], ascending=[True, False])                  
    .reset_index(drop=True)
)

In [193]:
#original

# rfr_comp_by_mileage_pivot = (
#     rfr_comp_by_mileage.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'
#         ],
#         columns='test_mileage',
#         values='size'
#     )
#     .sort_values(by='0-10000', ascending=False)
#     .fillna(0)
#     .reset_index()
# )
# rfr_comp_by_mileage_pivot.columns.name = None

# rfr_comp_by_mileage_pivot = filter_below_threshold(rfr_comp_by_mileage_pivot)
# rfr_comp_by_mileage_pivot.index = rfr_comp_by_mileage_pivot.index + 1
# rfr_comp_by_mileage_pivot_ranked = rank_vals(rfr_comp_by_mileage_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{rfr_comp_by_mileage_pivot_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs by mileage"))
#     display(rfr_comp_by_mileage_pivot_ranked.head(10))

In [194]:
rfr_comp_by_mileage_pivot = (
    rfr_comp_by_mileage.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'
        ],
        columns='test_mileage',
        values='size'
    )
    .sort_values(by='0-10000', ascending=False)
    .fillna(0)
    .reset_index()
)
rfr_comp_by_mileage_pivot.columns.name = None
rfr_comp_by_mileage_pivot = rfr_comp_by_mileage_pivot.reset_index(drop=True)
rfr_comp_by_mileage_pivot.index = rfr_comp_by_mileage_pivot.index + 1
rfr_comp_by_mileage_pivot_ranked = rank_cols(rfr_comp_by_mileage_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs by mileage"))
    display(rfr_comp_by_mileage_pivot_ranked.head(10))

Ranking of rfrs by mileage

,section_name,item_name,rfr_desc,0-10000,10000-30000,30000-60000,60000-90000,90000+
1,Tyres,Condition,Tyre tread depth not in accordance with the requirements,1,1,2,1,1
2,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,2,2,1,2,2
3,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",3,5,6,10,7
4,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,4,7,15,24,23
5,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,5,8,5,5,5
6,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",6,6,4,4,4
7,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",7,4,10,22,19
8,Identification of the vehicle,Registration plate,Number plate does not conform to the specified requirements,8,12,18,29,24
9,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,9,3,3,3,3
10,Audible warning (horn),Motorcycle audible warning (Horn),Audible warning not working,10,14,11,13,13


In [195]:
corr_df(rfr_comp_by_mileage_pivot)

,100%,75%,50%
0-10000,0.920050,0.837750,0.669458
90000+,0.937952,0.892811,0.824405
60000-90000,0.951138,0.909532,0.856935
10000-30000,0.958294,0.920969,0.863378
30000-60000,0.967965,0.936880,0.881370


In [196]:
js_dist_matrix = js_distance(rfr_comp_by_mileage_pivot)
font_size = 11
fig = px.imshow(
        js_dist_matrix,
        width=500,
        height=500
)
fig.update_layout(
    title=dict(
        text="JS Distance of rfrs by mileage",
        y=0.93,
        font=dict(size=15)
    )
)

fig.update_xaxes(tickfont=dict(size=font_size))
fig.update_yaxes(tickfont=dict(size=font_size))
fig.update_layout(
    coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
fig.show()

In [197]:
# js_dist_matrix = js_distance(rfr_comp_by_mileage_pivot)
# spearman_matrix = rfr_comp_by_mileage_pivot.corr(method='spearman', numeric_only=True)

# fig = make_subplots(
#     rows=1, cols=2,
#     subplot_titles=("JS Distance", "Spearman Correlation")
# )

# fig_js = px.imshow(
#         js_dist_matrix,
#         width=700,
#         height=700, 
# )
# fig_spearman = px.imshow(
#         spearman_matrix,
#         width=700,
#         height=700
# )

# fig.add_trace(fig_js.data[0], row=1, col=1)
# fig.add_trace(fig_spearman.data[0], row=1, col=2)

# fig.data[0].update(coloraxis="coloraxis1")
# fig.data[1].update(coloraxis="coloraxis2")

# fig.update_layout(
#     coloraxis1=dict(colorscale='Plasma', colorbar=dict(x=0.45, tickfont=dict(size=11))),
#     coloraxis2=dict(colorscale='Plasma_r', colorbar=dict(x=1.0, tickfont=dict(size=11))),
#     height=500, width=1100,
#     title_text="JS Distance vs Spearman Correlation of rfrs across mileage"
# )
# fig.update_xaxes(tickfont=dict(size=11))
# fig.update_yaxes(tickfont=dict(size=11))
# fig.update_yaxes(showticklabels=False, row=1, col=2)

# fig.show()

The tendency is similar here to decades correlation - the bigger gap between bins the lower correlation. "90000+" bin differs most from other on ranking and distribution. Notably that "60000-90000" ranking differs more from "90000+" than "0-100000". Ranks in the table above sorted by first bin (0-10000) shows 2 first rfrs have lead for all mileage bins.

### Rfrs comparison for different cylinder capacity

Next we create bins for different cylinder capacities and check does it have various rfrs.

In [198]:
first_test_failed_items_cc_bins = pd.cut(
    first_test_failed_items["cylinder_capacity"],
    bins=[40, 125, 250, 400, 500, 750, 1000, 1500, 2000, 2500],
    labels=["40–125", "125–250", "250–400", "400–500", "500–750", "750–1000", "1000–1500", "1500–2000", "2000–2500"],
    include_lowest=True,
    right=True
)

rfr_comp_by_cc = (
    first_test_failed_items.groupby(
        [first_test_failed_items_cc_bins, 'section_name', 'item_name', 'rfr_desc'], 
        observed=False, 
        as_index=False
    )
    .size()
    .loc[lambda x: x['size'] > 0]
    .sort_values(by=['cylinder_capacity', 'size'], ascending=[True, False])               
    .reset_index(drop=True)
)

In [199]:
#original

# rfr_comp_by_cc_pivot = (
#     rfr_comp_by_cc.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'
#         ],
#         columns='cylinder_capacity',
#         values='size'
#     )
#     .sort_values(by='40–125', ascending=False)
#     .fillna(0)
#     .reset_index()
# )
# rfr_comp_by_cc_pivot.columns.name = None
# rfr_comp_by_cc_pivot = filter_below_threshold(rfr_comp_by_cc_pivot)
# rfr_comp_by_cc_pivot.index = rfr_comp_by_cc_pivot.index + 1
# rfr_comp_by_cc_pivot_ranked = rank_vals(rfr_comp_by_cc_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{rfr_comp_by_cc_pivot_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs by cylinder capacity"))
#     display(rfr_comp_by_cc_pivot_ranked.head(10))

In [200]:
rfr_comp_by_cc_pivot = (
    rfr_comp_by_cc.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'
        ],
        columns='cylinder_capacity',
        values='size'
    )
    .sort_values(by='40–125', ascending=False)
    .fillna(0)
    .reset_index()
)
rfr_comp_by_cc_pivot.columns.name = None
rfr_comp_by_cc_pivot = rfr_comp_by_cc_pivot.reset_index(drop=True)
rfr_comp_by_cc_pivot.index = rfr_comp_by_cc_pivot.index + 1
rfr_comp_by_cc_pivot_ranked = rank_cols(rfr_comp_by_cc_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs by cylinder capacity"))
    display(rfr_comp_by_cc_pivot_ranked.head(10))

Ranking of rfrs by cylinder capacity

,section_name,item_name,rfr_desc,40–125,125–250,250–400,400–500,500–750,750–1000,1000–1500,1500–2000,2000–2500
1,Tyres,Condition,Tyre tread depth not in accordance with the requirements,1,8,1,2,1,1,1,1,1
2,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,2,2,2,1,2,2,2,10,5
3,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,3,9,3,4,3,4,3,3,2
4,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",4,1,4,8,11,10,7,12,9
5,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",5,13,13,3,8,13,20,40,12
6,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,6,10,9,7,9,9,6,11,4
7,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",7,6,6,5,4,6,8,6,5
8,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,8,3,4,6,5,5,9,5,7
9,Lamps and reflectors,Stop lamp,A stop lamp(s) does not illuminate by the operation of both brake controls or remains on when the brakes are released,9,12,12,9,14,15,14,9,7
10,Brakes,Roller brake test,Significant brake effort recorded with no brake applied indicating a binding brake,10,16,11,13,12,14,10,20,9


In [201]:
corr_df(rfr_comp_by_cc_pivot)

,100%,75%,50%
1500–2000,0.839933,0.713547,0.713697
2000–2500,0.842465,0.771876,0.680051
125–250,0.863897,0.723423,0.570610
40–125,0.883423,0.756498,0.757092
400–500,0.904543,0.812465,0.722883
250–400,0.921687,0.850532,0.810828
1000–1500,0.922410,0.856325,0.822381
500–750,0.923839,0.853822,0.811384
750–1000,0.933453,0.872756,0.829934


In [202]:
js_dist_matrix = js_distance(rfr_comp_by_cc_pivot)
font_size = 12
fig = px.imshow(
        js_dist_matrix,
        width=700,
        height=700
)
fig.update_layout(
    title=dict(
        text="JS Distance of rfrs by cylinder capacity",
        y=0.93,
        font=dict(size=18)
    )
)

fig.update_xaxes(tickfont=dict(size=font_size))
fig.update_yaxes(tickfont=dict(size=font_size))
fig.update_layout(
    coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
fig.show()

In [203]:
# js_dist_matrix = js_distance(rfr_comp_by_cc_pivot)
# spearman_matrix = rfr_comp_by_cc_pivot.corr(method='spearman', numeric_only=True)

# fig = make_subplots(
#     rows=1, cols=2,
#     subplot_titles=("JS Distance", "Spearman Correlation")
# )

# fig_js = px.imshow(
#         js_dist_matrix,
#         width=700,
#         height=700, 
# )
# fig_spearman = px.imshow(
#         spearman_matrix,
#         width=700,
#         height=700
# )

# fig.add_trace(fig_js.data[0], row=1, col=1)
# fig.add_trace(fig_spearman.data[0], row=1, col=2)

# fig.data[0].update(coloraxis="coloraxis1")
# fig.data[1].update(coloraxis="coloraxis2")

# fig.update_layout(
#     coloraxis1=dict(colorscale='Plasma', colorbar=dict(x=0.45, tickfont=dict(size=11))),
#     coloraxis2=dict(colorscale='Plasma_r', colorbar=dict(x=1.0, tickfont=dict(size=11))),
#     height=500, width=1100,
#     title_text="JS Distance vs Spearman Correlation of rfrs for different cylinder capacities"
# )
# fig.update_xaxes(tickfont=dict(size=11))
# fig.update_yaxes(tickfont=dict(size=11))
# fig.update_yaxes(showticklabels=False, row=1, col=2)

# fig.show()

As we see here, the mean rank correlation & js distance coefficient of cylinder capacity groups indicate that motorcycles with 2000-2500 cc differ the most in terms of ranks and rfr distribution. Worth notice the tendency is not linear, 2 leftmost and rightmost bins take first 4 places. The lowest rank correlation is between "2000-2500" and "125-250"cc (0.54) this pair has also highest js distance, the opposite is for "500-750" and "750-1000" bins (0.96 & 0.083). Looking at rank correlation heatmap we see that bins from "40-125" to "1000-1500" form a block with similar color, i.e their rank correlation is quite the same. Comparing rank correlation with js distance, column "2000-2500" is the second column where all values are negative, that is the magnitude/composition of rfrs is relatively similar between this cylinder capacity and other cc groups, but the rank order of those rfrs is comparatively more different.

### Rfrs across different areas

Now we analyse does different regions have the same rfrs or not.

In [204]:
first_test_failed_items_regions = first_test_failed_items['postcode_area'].map(postcode_region_dict)
rfr_by_region = (
    first_test_failed_items.groupby(
        [first_test_failed_items_regions, 'section_name', 
         'item_name', 'rfr_desc'], 
        as_index=False)
        .size()
        .sort_values(
            by=['postcode_area', 'size'], 
            ascending=[True, False])
        .reset_index(drop=True)
)

In [205]:
#original

# rfr_by_region_pivot = (
#     rfr_by_region.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'
#         ],
#         columns='postcode_area',
#         values='size'
#     )
#     .astype('Int64')
#     .sort_values(by='South West', ascending=False)
#     .fillna(0)
#     .reset_index()
# )
# rfr_by_region_pivot.columns.name = None
# rfr_by_region_pivot = filter_below_threshold(rfr_by_region_pivot)
# rfr_by_region_pivot.index = rfr_by_region_pivot.index + 1
# rfr_by_region_pivot_ranked = rank_vals(rfr_by_region_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{rfr_by_region_pivot_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs by UK regions"))
#     display(rfr_by_region_pivot_ranked.head(10))

In [206]:
rfr_by_region_pivot = (
    rfr_by_region.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'
        ],
        columns='postcode_area',
        values='size'
    )
    .astype('Int64')
    .sort_values(by='South West', ascending=False)
    .fillna(0)
    .reset_index()
)
rfr_by_region_pivot.columns.name = None
rfr_by_region_pivot = rfr_by_region_pivot.reset_index(drop=True)
rfr_by_region_pivot.index = rfr_by_region_pivot.index + 1
rfr_by_region_pivot_ranked = rank_cols(rfr_by_region_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs by UK regions"))
    display(rfr_by_region_pivot_ranked.head(10))

Ranking of rfrs by UK regions

,section_name,item_name,rfr_desc,East Midlands,East of England,Greater London,North East,North West,Scotland,South East,South West,Wales,West Midlands
1,Tyres,Condition,Tyre tread depth not in accordance with the requirements,1,2,1,2,1,2,1,1,1,2
2,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,2,1,3,1,2,1,2,2,2,1
3,Brakes,Brake pads,Brake lining or pad worn below 1.0mm,5,4,2,5,4,10,3,3,4,5
4,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,8,3,9,12,9,6,7,4,3,9
5,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",6,7,13,8,5,9,8,5,8,3
6,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",3,5,4,3,3,7,4,6,6,4
7,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",4,8,5,4,6,3,6,7,7,7
8,Lamps and reflectors,Rear reflectors,Reflector missing or reflecting white to the rear,7,6,14,7,8,8,5,8,5,6
9,Brakes,Roller brake test,Significant brake effort recorded with no brake applied indicating a binding brake,11,10,25,6,7,4,15,9,9,11
10,Identification of the vehicle,Registration plate,Number plate does not conform to the specified requirements,13,9,23,9,16,15,9,10,12,13


In [207]:
corr_df(rfr_by_region_pivot)

,100%,75%,50%
Greater London,0.906650,0.834916,0.788450
Scotland,0.967005,0.924207,0.866097
North East,0.974915,0.946733,0.909315
East of England,0.976047,0.948017,0.904366
West Midlands,0.976614,0.949853,0.908375
Wales,0.977756,0.952550,0.911984
East Midlands,0.978598,0.955859,0.930685
South West,0.979034,0.954287,0.919559
North West,0.979304,0.956158,0.920995
South East,0.980352,0.957785,0.924332


In [208]:
rfr_by_region_pivot.columns = rfr_by_region_pivot.columns.str.replace(' ', '<br>')
js_dist_matrix = js_distance(rfr_by_region_pivot)
font_size = 11
fig = px.imshow(
        js_dist_matrix,
        width=700,
        height=700
)
fig.update_layout(
    title=dict(
        text="JS Distance of rfrs by region cylinder capacity",
        y=0.93,
        font=dict(size=18)
    )
)

fig.update_xaxes(tickfont=dict(size=font_size))
fig.update_yaxes(tickfont=dict(size=font_size))
fig.update_layout(
    coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
fig.show()

In [209]:
# corr_df(rfr_by_region_pivot)
# fail_rate_by_region_cc['region'] = fail_rate_by_region_cc['region'].str.replace(' ', '<br>')
# js_dist_matrix = js_distance(rfr_by_region_pivot)
# font_size = 12
# fig = px.imshow(
#         js_dist_matrix,
#         width=900,
#         height=900
# )
# fig.update_layout(
#     title=dict(
#         text="JS Distance of rfrs by region cylinder capacity",
#         y=0.93,
#         font=dict(size=18)
#     )
# )

# fig.update_xaxes(tickfont=dict(size=font_size))
# fig.update_yaxes(tickfont=dict(size=font_size))
# fig.update_layout(
#     coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
# fig.show()

In [210]:
# js_dist_matrix = js_distance(rfr_by_region_pivot)
# spearman_matrix = rfr_by_region_pivot.corr(method='spearman', numeric_only=True)

# fig = make_subplots(
#     rows=1, cols=2,
#     subplot_titles=("JS Distance", "Spearman Correlation")
# )

# fig_js = px.imshow(
#         js_dist_matrix,
#         width=700,
#         height=700, 
# )
# fig_spearman = px.imshow(
#         spearman_matrix,
#         width=700,
#         height=700
# )

# fig.add_trace(fig_js.data[0], row=1, col=1)
# fig.add_trace(fig_spearman.data[0], row=1, col=2)

# fig.data[0].update(coloraxis="coloraxis1")
# fig.data[1].update(coloraxis="coloraxis2")

# fig.update_layout(
#     coloraxis1=dict(colorscale='Plasma', colorbar=dict(x=0.45, tickfont=dict(size=11))),
#     coloraxis2=dict(colorscale='Plasma_r', colorbar=dict(x=1.0, tickfont=dict(size=11))),
#     height=500, width=1100,
#     title_text="JS Distance vs Spearman Correlation of rfrs across different areas"
# )
# fig.update_xaxes(tickfont=dict(size=11))
# fig.update_yaxes(tickfont=dict(size=11))
# fig.update_yaxes(showticklabels=False, row=1, col=2)

# fig.show()

We clearly see from heatmap, JS distance matrix and average rank correlation by region, Greater London appears to differ the most from other regions, but overall the regions don't deviate much from each other. The defects "Tyre tread depth not in accordance with the requirements" and "A shock absorber not functioning or leaking severely" consistently rank within the top three across all regions.

### Honda CBR600F vs Yamaha FZS600  

Previously we saved two models of different brand which have similar cylinder capacity and comparably high number of motos which took the test, did they have different defects and did correlation deviate from brands' coefficient?

In [211]:
cbr_fzs_fail_rate = (first_test
    .loc[first_test['vehicle_id'].isin(honda_vs_yamaha)]
    .groupby('make')['passed']
    .agg(
        failed_pct=lambda x: (x == False).sum() / len(x) * 100,
        test_num='count'
        
    )
    .round(2)
    .reset_index()
)
cbr_fzs_fail_rate['failed_pct'] = cbr_fzs_fail_rate['failed_pct'].astype(str) + '%'
cbr_fzs_fail_rate

,make,failed_pct,test_num
0,HONDA,8.15%,4357
1,YAMAHA,9.04%,3306


In [212]:
first_test.loc[first_test['make'].isin(['HONDA', 'YAMAHA'])].groupby('make')['passed'].value_counts(normalize=True)

make    passed
HONDA   True      0.917174
        False     0.082826
YAMAHA  True      0.914287
        False     0.085713
Name: proportion, dtype: float64

As we see fail rate for CBR600F is lower than Honda's mean but for Yamaha is higher for this particular model. What about rfrs?

In [213]:
cbr_fzs_failed_items = (
    first_test_failed_items.loc[first_test_failed_items['vehicle_id'].isin(honda_vs_yamaha)]
        .groupby(
            ['make', 'section_name', 'item_name', 'rfr_desc'], 
            observed=False, 
            as_index=False
        )
        .size()
        .sort_values(by=['make', 'size'], ascending=[True, False])                
        .reset_index(drop=True)
)

In [214]:
#original

# cbr_fzs_pivot = (
#     cbr_fzs_failed_items.pivot(
#         index=[
#             'section_name', 
#             'item_name', 
#             'rfr_desc'
#         ],
#         columns='make',
#         values='size'
#     )
#     .astype('Int64')
#     .sort_values(by='HONDA', ascending=False)
#     .fillna(0)
#     .reset_index()
# )
# cbr_fzs_pivot.columns.name = None
# cbr_fzs_pivot = filter_below_threshold(cbr_fzs_pivot)
# cbr_fzs_pivot.index = cbr_fzs_pivot.index + 1
# cbr_fzs_pivot_ranked = rank_vals(cbr_fzs_pivot)
# with pd.option_context('display.max_colwidth', None):
#     display(Markdown(f"{cbr_fzs_pivot_ranked.shape[0]} of 303 rfrs take 80% of total"))
#     display(Markdown("Ranking of rfrs by cylinder capacity"))
#     display(cbr_fzs_pivot_ranked.head(10))

In [215]:
cbr_fzs_pivot = (
    cbr_fzs_failed_items.pivot(
        index=[
            'section_name', 
            'item_name', 
            'rfr_desc'
        ],
        columns='make',
        values='size'
    )
    .astype('Int64')
    .sort_values(by='HONDA', ascending=False)
    .fillna(0)
    .reset_index()
)
cbr_fzs_pivot.columns.name = None
cbr_fzs_pivot = cbr_fzs_pivot.reset_index(drop=True)
cbr_fzs_pivot.index = cbr_fzs_pivot.index + 1
cbr_fzs_pivot_ranked = rank_cols(cbr_fzs_pivot)
with pd.option_context('display.max_colwidth', None):
    display(Markdown("Ranking of rfrs by cylinder capacity"))
    display(cbr_fzs_pivot_ranked.head(10))

Ranking of rfrs by cylinder capacity

,section_name,item_name,rfr_desc,HONDA,YAMAHA
1,Suspension,Shock absorbers,A shock absorber not functioning or leaking severely,1,2
2,Lamps and reflectors,Stop lamp,"Stop lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",2,9
3,Tyres,Condition,Tyre tread depth not in accordance with the requirements,3,1
4,Lamps and reflectors,Individual direction indicators,"A direction indicator lamp missing, inoperative or in the case of a multiple light source more than 1/2 not functioning",4,8
5,Suspension,Wheel bearings,A wheel bearing with excessive play,5,17
6,Structure and attachments,Drive chain,"A transmission belt, chain, sprocket or pulley excessively loose or worn",6,6
7,Lamps and reflectors,Headlamp aim,The aim of a headlamp is not within limits laid down in the requirements,7,3
8,Brakes,Roller brake test,Excessive fluctuation in brake effort through each wheel revolution,7,12
9,Lamps and reflectors,Headlamp,"A headlamp missing, inoperative or more than ½ not functioning in the case of LED",8,11
10,Lamps and reflectors,Stop lamp,A stop lamp(s) does not illuminate by the operation of both brake controls or remains on when the brakes are released,9,10


In [216]:
corr_df(cbr_fzs_pivot)

,100%,75%,50%
HONDA,0.920944,0.880591,0.784362
YAMAHA,0.920944,0.880591,0.784362


In [217]:

# fail_rate_by_region_cc['region'] = fail_rate_by_region_cc['region'].str.replace(' ', '<br>')
# js_dist_matrix = js_distance(rfr_by_region_pivot)
# font_size = 12
# fig = px.imshow(
#         js_dist_matrix,
#         width=900,
#         height=900
# )
# fig.update_layout(
#     title=dict(
#         text="JS Distance of rfrs by region cylinder capacity",
#         y=0.93,
#         font=dict(size=18)
#     )
# )

# fig.update_xaxes(tickfont=dict(size=font_size))
# fig.update_yaxes(tickfont=dict(size=font_size))
# fig.update_layout(
#     coloraxis=dict(colorscale='Plasma', colorbar=dict(tickfont=dict(size=font_size))))
# fig.show()

In [218]:
js_distance_matrix = js_distance(cbr_fzs_pivot)
display(Markdown("Models' JS distance"))
js_distance_matrix

Models' JS distance

,HONDA,YAMAHA
HONDA,0.000000,0.295043
YAMAHA,0.295043,0.000000


In [219]:
display(Markdown("Brands' JS distance"))
brands_js_dist_matrix[['HONDA', 'YAMAHA']].head(2)

Brands' JS distance

,HONDA,YAMAHA
HONDA,0.000000,0.123098
YAMAHA,0.123098,0.000000


We see that correlation between two models is lower than correlation coefficient between two brands overall, and JS distance is higher for these specific models so despite having similar cylinder capacity, the models have slightly different defects and rfrs.

# 5. Conclusion

###  Disproportional number of tests in May
During 2024 we have unproportional number of tests, it was increasing since start of the year till end of the spring and consequently falled. The lowest fail rate is in May. This can be seasonal pattern, the best way to check previous data to confirm it. Low number of tests in winter, early spring/late autumn can be explained by high number of holiday days and weather conditions. In reverse, as we noticed before, having nearly 5% motorcycles that cover less than 200 miles per year that possibly can be explained by usage for leasure / second vehicle, we can assume some owners use motorcycles only in warm time of the year, accordingly, they take the test only when they use the vehicle. The reason for concern can be not only high VTS workload from May till August by itself, but possible impact on test result, meaning if we look just on numbers we can assume that it's easy to pass the test in high workload period. So some solution as advantages for completing the test not on workloaded season could take place. 
### Fail rate by region
As we said before, fail rate is higher on South of UK, and west parts of central England. We see that rural and coastal areas in the South-West tend to have higher fail rate on average. Possibly the residents ride seasonally or for leisure, so their bikes might sit unused for long stretches - which can lead to corrosion, tyre degradation, and brake issues that increase fail rates. By contrast, urban commuters often maintain regularly serviced bikes. The South-West is more exposed to rain, salty sea air, and humidity. Salt accelerates rusting and corrosion on metal components (brake lines, frames, exhausts), which are frequent MOT failure points. Damp conditions can also affect electrics and tyres. Due to lower population, possibly there are fewer number of large motorcycle dealerships or service centers there, riders may be more likely to do their own maintenance or delay professional servicing - which can lead to higher fail rates at formal MOT testing. Roads in some parts of the South-West are narrow, twisty, and rural, often with more debris, mud, and uneven surfaces. That can lead to wear and tear on tyres and suspension, and also cause damage that’s not obvious until MOT inspection. My suggestion is to warn owners who store their vehicle in such regions to pay attention on storing conditions, use some addional treatment during/after season of usage. As an example we have intersection of different groups, for example for 40-125 cc if they located in South-West, their fail rate would be even higher than average, in the same time London is suitable for such type of motorcycles, it has the lowest fail rate of all regions. Meanwhile difference between rfrs ranks for various regions is almost absent, the region with highest fail rate has one of the highest rank correlation with other regions.
### Fail rate is lower for vehicles registered before 1993
Looking at fail rate by date of registration we see that the measure is comparably low for vehicles registrated before 1993. For some years it's even lower than for new vehicles registrated in 2017-2021. We can say that it's a cluster of collectible vehicles that are not used at all or used occassionally and are well-treated. The same situation takes place for most popular brands, notoriously
, it's not the case for Harley-Davidson.
### Fail rate by mileage
Obviously it's getting higher with higher mileage. But it's not the case for some brands(Suzuki, Kawasaki, Harley-Davidson), They form group of well-treated / collectible motorcycles.
### Fail rate by brand
Average fail rate for all motorcycles - 7.63%. From bar chart it's clearly that there are some clusters with similar fail rate.
### Fail rate by cylinder capacity
Cylinder capacity group "40-125" has highest fail rate among all cc. It almost twice as higher than second rank, "500-750". The higher the cylinder capacity, the less likely the motorcycle is to fail the test. Higher-capacity motorcycles are generally more expensive and built with higher-quality components and it explicitly influences on test result.
### Fail rate by brand and cylinder capacity 
Highest impact on fail rate has brand and cylinder capacity connection. It's proven by correlation coefficient for fail rate and brands that took minimum 1000 tests. Top brands repeat the pattern.

### Fail rate by region and cylinder capacity 
i would say regions here perform as accelerant for fail rate, i.e for our cc with highest fail rate, 40-125, if the region has high fail rate, fail rate for cc will get higher, as we see for South-West, and opposite situation for Greater London. For 6 out of 9 cc bins fail rate spikes are in a range of 2% for all years.

### Fail rate by cylinder capacity and decade of registration

### Number of items by category and code
#### Rfr deficiency categories
According to [official MOT website](https://www.gov.uk/guidance/mot-inspection-manual-for-motorcycles/introduction), various defect categories may be available for the same item depending on the nature or severity of the defect – minor, major or dangerous. Just looking at columns filling we can conclude the less count "Minor" category has per item, the more severe item defect could be.

##### Rfr deficiency category "Pre-EU directive"
From our table we see that this category has its own section "Non-component advisories" and has no rfr type code except for "A". If we check rfr description for this section or item it returns "np.nan". We can conclude from the title that it's some common advices about vehicle conditions.

##### Rfr deficiency category "Minor"
Many items have 0 for both rfr type codes, meaning that severity of damage is higher or nature is different. Only "Audible warning (horn)" section has non-zero count for "A" in the category.

##### Rfr_type_code "A" & "M" 
if the item has "A" for "Dangerous" rfr deficiency category it won't have "M" for "Minor" but the item has "A" for "Major" it can have "A" for "Minor". Possibly it can be explained that nature of the damage is closer between "Minor" & "Major" categories.

##### Rfr_type_code "PRS"
Only 4 items in "Major" and 2 items in "Major" category have more "PRS" count than "F", meaning they are easy to fix despite referring to more "severe" category.

##### Rfr_type_code "F" for "Dangerous" category
For all sections we have one item that has much higher count than others.

##### Rfr_type_code "F" for "Major" category
As well as "F" for "Dangerous" category, "Major" category has one leader, except for "Lamp and reflectors" sections, high count items are distributed more uniformly.

#### Sections
Brakes, steering, structure and attachments, suspension, tyres and wheels are categories that almost have no count for "Minor" rfr deficiency category, so the damage is more severe for them. Lamps and reflectors is the only section where values for rfr type code "F" has uniformly distibuted count. For other sections there are 1-3 huge count and others are small ones. Lamps and reflectors also has only one item for "Dangerous" category - stop lamp, as we noticed before, high number of "PRS" for this section means that most of the items are easy and fast to fix.
### Rfrs
For failed vehicles, items that belong to "Dangerous" and "Major" `['rfr_deficiency_category']` and has "F" (cause failing the test) take 67.3%. "Lamp and reflectors" section mostly takes first place for "F" & "PRS". On average vehicles that failed the test have 2 parts that cause the failure. About 40% of the failed parts belongs to "Lamps and reflectors", "Audible warning (horn)" and "Identification of the vehicle" and are relatively easy to fix. Their high prevalence suggests a significant opportunity for owners to conduct routine, preventative maintenance checks. Meanwhile parts that belong to "Suspension", "Brake" or "Steering" sections require specialized tools (e.g., bearing drivers, shock recharge equipment), deep mechanical knowledge, and precision component installation to ensure the structural integrity and stable handling of the motorcycle are restored safely. 20 of 303 rfrs take 61% of total, 9 of 20 belongs to "Lamp and reflectors" section.

### Dive deeper in vehicles that didn't try to retest after failure
About 6k motorcycles didn't retake the test after failure. My suggestion is communicate with vehicles' owners to find out reasons of such decision and add the reasons to statistics. As comparison of rfrs showed it seems like not due to specific rfrs that make it impossible to retake the test. Most defects belong to the "Major" category, and 5 of the top 10 rfrs fall within the "Lamps and reflectors" section. The only notable rank difference appears for the defect "A transmission belt, chain, sprocket, or pulley excessively loose or worn." However, rank correlation coefficient shows that there is very small difference between the vehicles that retook the test after failure and vehicles that decided not to retake the test.

### Negative trend of increasing share of "Dangerous" rfrs.
For period of vehicle registration from 1980 till now, we have negative trend with increasing number of rfrs that belong to "Dangerous" category. Aproximately it was increased twice, from 13% to 26%.
 We have some clusters, i.e vehicles with quite low fail rate has higher share of "Dangerous" parts. Notably, "more popular brands" have higher share of "Dangerous" parts. For mileage that comes along with vehicles age, we haven't got enough evidence to say the difference between share of dangerous parts for various mileage bins is significant.

#### Rfrs comparsion for various categories
For all tables we filter out rfrs which take less than 0.1% per group to reduce the impact of small shares, that take most of all rfrs, then calculate rank correlation to find out does groups ranks placement differ. We could get different results if we won't filter shares/use another threshold/define another group sizes, however using exactly correlation will show the same direction.
Correlation between groups are quite high even for so to call opposite groups, for example 125-250 and 2000-2500 cc are 0.54, Ducati and Piaggio (0.57), 1980-1990 and 2020-2021 (0.71). For some groups, as regions correlation is very high. The reason is that 20 of 303 rfrs take 61% of total. My opinion is overall vehicles fails the test not because of specific rfrs, but other reasons, as we said before, vehicles with low cc is prone to damage more, as well it depends on brand. If it's vehicle that were registrated before 1990.